# Pipeline de datos masivos: medicamentos vigentes en Colombia (CUM - INVIMA)
**Gestión de Datos Masivos G002 · Institución Universitaria Pascual Bravo · 2026-2**
**Integrantes:** Martín Sánchez Urrego · Samuel Montoya Garzón

| Aspecto | Detalle |
|---|---|
| Fuente | [Código Único de Medicamentos vigentes (INVIMA)](https://www.datos.gov.co/resource/i7cb-raxc.json), datos.gov.co, API Socrata |
| Tamaño | ~156.000 registros, actualización frecuente (última: nov 2026) |
| Columnas de fecha | `fechaexpedicion`, `fechavencimiento`, `fechaactivo`, `fechainactivo` |
| **id_registro** | Campo de sistema Socrata **`:id`** (único por fila). La fuente no tiene llave natural: ni `expediente + consecutivocum + principioactivo + nombrerol + tiporol` es única |
| Marca de agua | Campo de sistema **`:updated_at`** (última actualización de cada registro) |
| Arquitectura | **Lambda**: batch (Spark → Parquet → SQLite) + capa rápida (Kafka → Structured Streaming) |

### Pregunta principal
> ¿Cómo está compuesta la oferta de medicamentos con registro vigente en Colombia (por grupo
> terapéutico ATC, vía de administración y origen fabricante o importador), y qué tanto depende de
> importaciones y de pocos titulares?

**¿Quién usaría los indicadores?** El Ministerio de Salud y los analistas de política farmacéutica
(riesgo de desabastecimiento por dependencia de importaciones), EPS e IPS (planeación de compras) y
cadenas de droguerías.

### Arquitectura por zonas
![diagrama](https://mermaid.ink/img/Zmxvd2NoYXJ0IExSCiAgQVsiQVBJIFNvY3JhdGEgZGF0b3MuZ292LmNvPGJyLz5QeXRob24gcmVxdWVzdHMiXSAtLT4gQlsiWk9OQSBSQVc8YnIvPkpTT05MIHNvYnJlIHBvciBmZWNoYV9pbmdlc3RhPGJyLz4rIG1hbmlmaWVzdG8iXQogIEIgLS0-IENbIkFwYWNoZSBLYWZrYTxici8-dG9waWNvIGN1bV9tZWRpY2FtZW50b3M8YnIvPjMgcGFydGljaW9uZXMsIGtleT1pZF9yZWdpc3RybyJdCiAgQyAtLT4gRFsiQ29uc3VtaWRvciBncnVwbyBjdW0taGRmczxici8-SERGUyAvZGF0YWxha2UvcmF3PGJyLz4xIGFyY2hpdm8gcG9yIGxvdGUiXQogIEQgLS0-IEVbIk1hcFJlZHVjZSBQeXRob248YnIvPmNvbnRlbyBwb3IgQVRDIHZzIFNwYXJrIl0KICBEIC0tPiBGWyJQeVNwYXJrPGJyLz5aT05BIENVUkFEQSBQYXJxdWV0PGJyLz5wYXJ0aXRpb25CeSBhdGNfbml2ZWwxIl0KICBGIC0tPiBHWyJDYWxpZGFkICsgbWFyY2EgZGUgYWd1YTxici8-aWRlbXBvdGVuY2lhIl0KICBHIC0tPiBIWyJaT05BIFNFUlZJQ0lPPGJyLz5TUUxpdGUgaW5kaWNhZG9yZXMiXQogIEMgLS0-IElbIkNBUEEgUkFQSURBPGJyLz5TdHJ1Y3R1cmVkIFN0cmVhbWluZzxici8-dmVudGFuYSArIHdhdGVybWFyayArIGNoZWNrcG9pbnQiXQogIEkgLS0-IEgKICBIIC0tPiBKWyJUYWJsZXJvIFBsb3RseSBIVE1MIl0=)

**¿Por qué Lambda?** La fuente es un *snapshot* que se descarga por lotes desde la API: la vista batch
(deduplicada, limpia y validada) es la verdad. Además, cada registro viaja por Kafka y una capa rápida
con Structured Streaming da conteos en ventanas de tiempo a medida que llegan los mensajes. La zona de
servicio une ambas vistas (`v_lambda_tiporol`). Una arquitectura Kappa obligaría a reprocesar todo
desde el log de Kafka para corregir datos, y aquí la corrección (deduplicar, tipar) es más natural en batch.

## 0. Preparación del entorno (Google Colab)
Se instalan Java 11, Kafka 3.7 (KRaft, sin ZooKeeper) y Hadoop 3.3.6 pseudo-distribuido, como en los
laboratorios. La primera vez tarda unos 5 minutos (descarga de Hadoop).

In [1]:
import os
BASE = "/content/pp-cum-invima"
os.makedirs(f"{BASE}/src/mapreduce", exist_ok=True)
os.makedirs(f"{BASE}/evidencias", exist_ok=True)
os.chdir(BASE)
os.environ.update({
    "JAVA_HOME": "/usr/lib/jvm/java-11-openjdk-amd64",
    "HADOOP_HOME": "/opt/hadoop",
    "KAFKA_HOME": "/opt/kafka",
    "PP_BASE": BASE,
    "PYSPARK_PYTHON": "python3",
})
os.environ["PATH"] = ("/opt/hadoop/bin:/opt/kafka/bin:/usr/lib/jvm/java-11-openjdk-amd64/bin:"
                      + os.environ["PATH"])
print(os.getcwd())

/content/pp-cum-invima


In [2]:
%%writefile requirements.txt
pyspark==3.5.3
confluent-kafka>=2.4
requests>=2.31
pandas>=2.0
plotly>=5.20

Writing requirements.txt


In [3]:
%%writefile setup_colab.sh
#!/usr/bin/env bash
# Prepara Google Colab con Java 11, Apache Kafka 3.7 (modo KRaft, sin ZooKeeper) y
# Hadoop 3.3.6 en modo pseudo-distribuido (HDFS: 1 NameNode + 1 DataNode).
# Es idempotente: si algo ya está instalado o corriendo, no lo repite.
set -e
KAFKA_VER=3.7.2
SCALA_VER=2.13
HADOOP_VER=3.3.6
OPT=/opt
export JAVA_HOME=/usr/lib/jvm/java-11-openjdk-amd64
export HADOOP_HOME=$OPT/hadoop
export KAFKA_HOME=$OPT/kafka

# ---------- Java 11 ----------
if [ ! -d "$JAVA_HOME" ]; then
  echo ">> Instalando Java 11..."
  apt-get -qq update > /dev/null
  apt-get -qq install -y openjdk-11-jdk-headless > /dev/null
fi

descargar () {   # descargar <archivo> <url1> <url2> ...
  local destino=$1; shift
  [ -s "$destino" ] && return 0
  for url in "$@"; do
    echo "   descargando $url"
    if wget -q --timeout=60 "$url" -O "$destino"; then return 0; fi
    rm -f "$destino"
  done
  echo "No se pudo descargar $destino"; return 1
}

cd $OPT
# ---------- Kafka ----------
if [ ! -d "$KAFKA_HOME" ]; then
  echo ">> Descargando Kafka $KAFKA_VER..."
  descargar kafka.tgz \
    "https://archive.apache.org/dist/kafka/$KAFKA_VER/kafka_$SCALA_VER-$KAFKA_VER.tgz" \
    "https://dlcdn.apache.org/kafka/$KAFKA_VER/kafka_$SCALA_VER-$KAFKA_VER.tgz"
  tar xzf kafka.tgz && mv kafka_$SCALA_VER-$KAFKA_VER kafka && rm kafka.tgz
fi

# ---------- Hadoop ----------
if [ ! -d "$HADOOP_HOME" ]; then
  echo ">> Descargando Hadoop $HADOOP_VER (unos 700 MB, puede tardar unos minutos)..."
  descargar hadoop.tgz \
    "https://dlcdn.apache.org/hadoop/common/hadoop-$HADOOP_VER/hadoop-$HADOOP_VER.tar.gz" \
    "https://archive.apache.org/dist/hadoop/common/hadoop-$HADOOP_VER/hadoop-$HADOOP_VER.tar.gz"
  tar xzf hadoop.tgz && mv hadoop-$HADOOP_VER hadoop && rm hadoop.tgz
fi

# ---------- Configuración HDFS pseudo-distribuido ----------
cat > $HADOOP_HOME/etc/hadoop/core-site.xml <<EOF
<configuration>
  <property><name>fs.defaultFS</name><value>hdfs://localhost:9000</value></property>
  <property><name>hadoop.tmp.dir</name><value>/opt/hdfs_data/tmp</value></property>
</configuration>
EOF
cat > $HADOOP_HOME/etc/hadoop/hdfs-site.xml <<EOF
<configuration>
  <property><name>dfs.replication</name><value>1</value></property>
  <property><name>dfs.namenode.name.dir</name><value>file:///opt/hdfs_data/nn</value></property>
  <property><name>dfs.datanode.data.dir</name><value>file:///opt/hdfs_data/dn</value></property>
  <property><name>dfs.permissions.enabled</name><value>false</value></property>
</configuration>
EOF
grep -q "PP_CONFIG" $HADOOP_HOME/etc/hadoop/hadoop-env.sh || cat >> $HADOOP_HOME/etc/hadoop/hadoop-env.sh <<EOF
# PP_CONFIG
export JAVA_HOME=$JAVA_HOME
export HDFS_NAMENODE_USER=root
export HDFS_DATANODE_USER=root
export HDFS_SECONDARYNAMENODE_USER=root
EOF

puerto_abierto () { (echo > /dev/tcp/localhost/$1) >/dev/null 2>&1; }

if [ ! -d /opt/hdfs_data/nn/current ]; then
  echo ">> Formateando el NameNode..."
  $HADOOP_HOME/bin/hdfs namenode -format -force -nonInteractive > /dev/null 2>&1
fi
if ! puerto_abierto 9000; then
  echo ">> Iniciando HDFS (NameNode + DataNode)..."
  $HADOOP_HOME/bin/hdfs --daemon start namenode
  $HADOOP_HOME/bin/hdfs --daemon start datanode
fi
for i in $(seq 1 30); do puerto_abierto 9000 && break; sleep 2; done
$HADOOP_HOME/bin/hdfs dfsadmin -safemode wait > /dev/null 2>&1 || true

# ---------- Kafka en modo KRaft ----------
KCFG=$KAFKA_HOME/config/kraft/server.properties
sed -i 's#^log.dirs=.*#log.dirs=/opt/kafka_data#' $KCFG
if [ ! -f /opt/kafka_data/meta.properties ]; then
  echo ">> Formateando el almacenamiento de Kafka (KRaft)..."
  UUID=$($KAFKA_HOME/bin/kafka-storage.sh random-uuid)
  $KAFKA_HOME/bin/kafka-storage.sh format -t "$UUID" -c $KCFG > /dev/null
fi
if ! puerto_abierto 9092; then
  echo ">> Iniciando el broker de Kafka..."
  KAFKA_HEAP_OPTS="-Xmx1G -Xms512M" $KAFKA_HOME/bin/kafka-server-start.sh -daemon $KCFG
fi
for i in $(seq 1 30); do puerto_abierto 9092 && break; sleep 2; done

echo ""
echo ">> Procesos Java activos (jps):"
$JAVA_HOME/bin/jps
puerto_abierto 9000 && echo "HDFS  OK (hdfs://localhost:9000)" || echo "HDFS  NO RESPONDE"
puerto_abierto 9092 && echo "Kafka OK (localhost:9092)" || echo "Kafka NO RESPONDE"

Writing setup_colab.sh


In [4]:
!bash setup_colab.sh
!pip install -q -r requirements.txt

>> Descargando Kafka 3.7.2...
>> Descargando Hadoop 3.3.6 (unos 700 MB, puede tardar unos minutos)...
>> Formateando el NameNode...
>> Iniciando HDFS (NameNode + DataNode)...
>> Formateando el almacenamiento de Kafka (KRaft)...
>> Iniciando el broker de Kafka...

>> Procesos Java activos (jps):
1842 NameNode
1975 DataNode
2318 Kafka
2401 Jps
HDFS  OK (hdfs://localhost:9000)
Kafka OK (localhost:9092)


Módulos compartidos: configuración, utilidades y sesión de Spark con el esquema del sobre.

In [5]:
%%writefile src/config.py
"""Configuración central del pipeline CUM (INVIMA).

Todas las rutas y parámetros viven aquí para que cada script use exactamente
los mismos valores (fuente única de verdad).
"""
import os

# Carpeta raíz del proyecto (un nivel arriba de src/)
BASE_DIR = os.environ.get(
    "PP_BASE", os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
)

# ---------------- Fuente ----------------
FUENTE = "datos.gov.co"
DATASET = "i7cb-raxc"
NOMBRE_DATASET = "Código Único de Medicamentos (CUM) vigentes - INVIMA"
API_URL = f"https://www.datos.gov.co/resource/{DATASET}.json"
META_URL = f"https://www.datos.gov.co/api/views/{DATASET}.json"
PAGE_SIZE = 50000          # máximo recomendado por Socrata por página
MAX_REINTENTOS = 5
TIMEOUT_S = 120
# Token opcional de Socrata (sube el límite de peticiones). No es obligatorio.
APP_TOKEN = os.environ.get("SOCRATA_APP_TOKEN")

# Campo que identifica cada registro sin repetirse (campo de sistema Socrata)
CAMPO_ID = ":id"
# Campo usado como marca de agua (última fecha de actualización del registro)
CAMPO_WATERMARK = ":updated_at"

# ---------------- Zonas locales ----------------
DATA_DIR = os.path.join(BASE_DIR, "data")
RAW_DIR = os.path.join(DATA_DIR, "raw", FUENTE, DATASET)
MANIFEST_DIR = os.path.join(DATA_DIR, "manifiestos")
STATE_DIR = os.path.join(DATA_DIR, "state")
WATERMARK_FILE = os.path.join(STATE_DIR, "watermark.json")
PUBLICADOS_FILE = os.path.join(STATE_DIR, "archivos_publicados.json")
CALIDAD_DIR = os.path.join(DATA_DIR, "calidad")
SERVICIO_DIR = os.path.join(DATA_DIR, "servicio")
SERVICE_DB = os.path.join(SERVICIO_DIR, "servicio.db")
EVIDENCIAS_DIR = os.path.join(BASE_DIR, "evidencias")
TMP_DIR = os.path.join(DATA_DIR, "tmp")

# ---------------- Kafka ----------------
KAFKA_BOOTSTRAP = os.environ.get("KAFKA_BOOTSTRAP", "localhost:9092")
TOPIC = "cum_medicamentos"
PARTICIONES = 3
GROUP_ID = "cum-hdfs"
LOTE_HDFS = 20000          # mensajes por archivo en HDFS (un archivo por lote)
IDLE_SEGUNDOS = 15         # el consumidor termina si no llegan mensajes en este tiempo

# ---------------- HDFS ----------------
HDFS_URI = os.environ.get("HDFS_URI", "hdfs://localhost:9000")
HDFS_RAW = f"/datalake/raw/{FUENTE}/{DATASET}"
HDFS_CURADA = "/datalake/curada/cum"
HDFS_DESCARTADOS = "/datalake/descartados/cum"
HDFS_MR_OUT = "/datalake/mapreduce/conteo_atc"
HDFS_CHECKPOINT = "/checkpoints/cum_stream"
HDFS_STREAM_OUT = "/datalake/stream/conteos_ventana"

# ---------------- Spark ----------------
SPARK_VERSION = "3.5.3"
KAFKA_PACKAGE = f"org.apache.spark:spark-sql-kafka-0-10_2.12:{SPARK_VERSION}"

# Nombres de los grupos ATC nivel 1 (OMS)
ATC_NIVEL1 = {
    "A": "Tracto alimentario y metabolismo",
    "B": "Sangre y órganos hematopoyéticos",
    "C": "Sistema cardiovascular",
    "D": "Dermatológicos",
    "G": "Sistema genitourinario y hormonas sexuales",
    "H": "Hormonas sistémicas",
    "J": "Antiinfecciosos sistémicos",
    "L": "Antineoplásicos e inmunomoduladores",
    "M": "Sistema musculoesquelético",
    "N": "Sistema nervioso",
    "P": "Antiparasitarios",
    "R": "Sistema respiratorio",
    "S": "Órganos de los sentidos",
    "V": "Varios",
    "SIN_ATC": "Sin código ATC válido",
}

for _d in (RAW_DIR, MANIFEST_DIR, STATE_DIR, CALIDAD_DIR, SERVICIO_DIR,
           EVIDENCIAS_DIR, TMP_DIR):
    os.makedirs(_d, exist_ok=True)

Writing src/config.py


In [6]:
%%writefile src/utils.py
"""Funciones auxiliares compartidas: JSON, HDFS por línea de comandos y evidencias."""
import glob
import json
import os
import subprocess
from datetime import datetime
from zoneinfo import ZoneInfo

import config as C

TZ = ZoneInfo("America/Bogota")


def ahora():
    return datetime.now(TZ)


def leer_json(ruta, defecto=None):
    if os.path.exists(ruta):
        with open(ruta, encoding="utf-8") as f:
            return json.load(f)
    return defecto


def escribir_json(ruta, obj):
    os.makedirs(os.path.dirname(ruta), exist_ok=True)
    with open(ruta, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2, default=str)


def guardar_evidencia(nombre, contenido):
    """Guarda un texto o un objeto JSON en la carpeta evidencias/."""
    ruta = os.path.join(C.EVIDENCIAS_DIR, nombre)
    if isinstance(contenido, (dict, list)):
        escribir_json(ruta, contenido)
    else:
        with open(ruta, "w", encoding="utf-8") as f:
            f.write(str(contenido))
    return ruta


# ---------------- HDFS (vía CLI, como en el laboratorio) ----------------
def _hdfs_bin():
    home = os.environ.get("HADOOP_HOME")
    return os.path.join(home, "bin", "hdfs") if home else "hdfs"


def hdfs(*args, check=True, capturar=True):
    """Ejecuta `hdfs dfs <args>` y devuelve la salida estándar."""
    cmd = [_hdfs_bin(), "dfs", *args]
    r = subprocess.run(cmd, capture_output=capturar, text=True)
    if check and r.returncode != 0:
        raise RuntimeError(f"Falló {' '.join(cmd)}:\n{r.stderr}")
    return r.stdout if capturar else ""


def hdfs_existe(ruta):
    return subprocess.run([_hdfs_bin(), "dfs", "-test", "-e", ruta]).returncode == 0


def hdfs_du_bytes(ruta):
    """Tamaño total (bytes) de una ruta HDFS."""
    if not hdfs_existe(ruta):
        return 0
    salida = hdfs("-du", "-s", ruta).strip()
    return int(salida.split()[0]) if salida else 0


def hdfs_contar_lineas(patron):
    """Cuenta líneas de los archivos que cumplen el patrón (glob HDFS)."""
    p1 = subprocess.Popen([_hdfs_bin(), "dfs", "-cat", patron],
                          stdout=subprocess.PIPE, stderr=subprocess.DEVNULL)
    n = sum(1 for _ in p1.stdout)
    p1.wait()
    return n


# ---------------- Zona raw local ----------------
def archivos_raw_locales():
    return sorted(glob.glob(os.path.join(C.RAW_DIR, "fecha_ingesta=*", "*.jsonl")))


def ultimo_manifiesto():
    ruta = leer_json(os.path.join(C.STATE_DIR, "ultimo_manifiesto.json"), {}).get("ruta")
    return leer_json(ruta) if ruta else None


def tamano_dir_local(ruta):
    total = 0
    for raiz, _, archivos in os.walk(ruta):
        for a in archivos:
            total += os.path.getsize(os.path.join(raiz, a))
    return total


def mb(b):
    return round(b / 1024 / 1024, 2)

Writing src/utils.py


In [7]:
%%writefile src/spark_utils.py
"""Sesión de Spark y esquema del «sobre» raw, compartidos por los scripts Spark."""
from pyspark.sql import SparkSession
from pyspark.sql.types import StringType, StructField, StructType

import config as C

# Campos del registro original (payload). Se leen todos como texto: la zona raw
# no corrige tipos; eso se hace en la zona curada.
CAMPOS_PAYLOAD = [
    ":id", ":created_at", ":updated_at",
    "expediente", "producto", "titular", "registrosanitario", "fechaexpedicion",
    "fechavencimiento", "estadoregistro", "expedientecum", "consecutivocum",
    "cantidadcum", "descripcioncomercial", "estadocum", "fechaactivo",
    "fechainactivo", "muestramedica", "unidad", "atc", "descripcionatc",
    "viaadministracion", "concentracion", "principioactivo", "unidadmedida",
    "cantidad", "unidadreferencia", "formafarmaceutica", "nombrerol", "tiporol",
    "modalidad", "ium",
]

ESQUEMA_PAYLOAD = StructType([StructField(c, StringType()) for c in CAMPOS_PAYLOAD])

ESQUEMA_SOBRE = StructType([
    StructField("fuente", StringType()),
    StructField("dataset", StringType()),
    StructField("id_registro", StringType()),
    StructField("fecha_ingesta", StringType()),
    StructField("ts_ingesta", StringType()),
    StructField("run_id", StringType()),
    StructField("payload", ESQUEMA_PAYLOAD),
])


def get_spark(app, kafka=False):
    b = (SparkSession.builder.appName(app).master("local[*]")
         .config("spark.driver.memory", "4g")
         .config("spark.sql.shuffle.partitions", "8")
         .config("spark.hadoop.fs.defaultFS", C.HDFS_URI)
         .config("spark.sql.session.timeZone", "America/Bogota")
         .config("spark.sql.legacy.timeParserPolicy", "CORRECTED")
         .config("spark.sql.ansi.enabled", "false")
         .config("spark.ui.showConsoleProgress", "false"))
    if kafka:
        b = b.config("spark.jars.packages", C.KAFKA_PACKAGE)
    spark = b.getOrCreate()
    spark.sparkContext.setLogLevel("ERROR")
    return spark


def leer_raw_hdfs(spark):
    """Lee todos los sobres JSONL del data lake raw en HDFS."""
    return spark.read.schema(ESQUEMA_SOBRE).json(f"{C.HDFS_URI}{C.HDFS_RAW}")

Writing src/spark_utils.py


## Fase 2. Ingesta a la zona raw
- Descarga por la API Socrata con **paginación** (`$limit=50000` + `$offset`, `$order=:id` para que el
  orden sea estable) y **reintentos** con espera exponencial ante 429/5xx o caídas de red.
- Cada registro se guarda **intacto** en `payload` dentro de un **sobre** con `fuente`, `dataset`,
  `id_registro`, `fecha_ingesta`, `ts_ingesta` y `run_id`.
- Archivos JSON Lines particionados por `fecha_ingesta=AAAA-MM-DD`.
- Cada ejecución deja un **manifiesto** (URL, parámetros, páginas, conteo, marca de agua).
- **Incremental:** si existe `data/state/watermark.json`, solo se piden registros con `:updated_at` mayor.
  La marca de agua se confirma únicamente cuando la zona curada pasa la calidad (fase 5/6).

In [8]:
%%writefile src/01_ingesta.py
"""Fase 2 - Ingesta a la zona raw.

Descarga el dataset CUM desde la API Socrata de datos.gov.co con paginación y
reintentos, envuelve cada registro en un «sobre» JSON sin modificarlo y lo guarda
en JSON Lines particionado por fecha_ingesta=AAAA-MM-DD. Cada ejecución deja un
manifiesto con la URL, los parámetros, las páginas y el conteo.

Carga incremental: si existe una marca de agua (último :updated_at cargado con
éxito) solo se piden los registros con :updated_at posterior.

Uso:
    python src/01_ingesta.py            # incremental (o completa la primera vez)
    python src/01_ingesta.py --completa # ignora la marca de agua
    python src/01_ingesta.py --limite 1000  # prueba rápida
"""
import argparse
import hashlib
import json
import os
import sys
import time

import requests

import config as C
from utils import ahora, escribir_json, leer_json


def leer_watermark():
    return leer_json(C.WATERMARK_FILE, {}).get("watermark")


def get_con_reintentos(sesion, params):
    """GET con reintentos y espera exponencial ante errores transitorios."""
    espera = 2
    for intento in range(1, C.MAX_REINTENTOS + 1):
        try:
            r = sesion.get(C.API_URL, params=params, timeout=C.TIMEOUT_S)
            if r.status_code == 200:
                return r.json(), intento
            if r.status_code in (429, 500, 502, 503, 504):
                print(f"   HTTP {r.status_code}: reintento {intento}/{C.MAX_REINTENTOS} en {espera}s")
            else:
                r.raise_for_status()  # error 4xx no transitorio: no tiene sentido reintentar
        except (requests.ConnectionError, requests.Timeout) as e:
            print(f"   {type(e).__name__}: reintento {intento}/{C.MAX_REINTENTOS} en {espera}s")
        time.sleep(espera)
        espera *= 2
    raise RuntimeError(f"La API no respondió tras {C.MAX_REINTENTOS} intentos")


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--completa", action="store_true", help="ignorar la marca de agua")
    ap.add_argument("--limite", type=int, default=None, help="máximo de registros (pruebas)")
    args = ap.parse_args()

    inicio = ahora()
    run_id = inicio.strftime("%Y%m%dT%H%M%S")
    fecha_ingesta = inicio.strftime("%Y-%m-%d")
    watermark = None if args.completa else leer_watermark()

    # SoQL exige que el * vaya primero; los campos de sistema (:id, :updated_at)
    # no vienen por defecto y hay que pedirlos explícitamente.
    params_base = {"$select": f"*,{C.CAMPO_ID},:created_at,{C.CAMPO_WATERMARK}"}
    if watermark:
        # la API devuelve '...Z' pero el filtro SoQL espera el timestamp sin la Z
        params_base["$where"] = f"{C.CAMPO_WATERMARK} > '{watermark.rstrip('Z')}'"
        params_base["$order"] = f"{C.CAMPO_WATERMARK},{C.CAMPO_ID}"
        print(f"Carga INCREMENTAL: registros con {C.CAMPO_WATERMARK} > {watermark}")
    else:
        params_base["$order"] = C.CAMPO_ID  # orden estable para paginar sin saltos
        print("Carga COMPLETA (no hay marca de agua)")

    sesion = requests.Session()
    if C.APP_TOKEN:
        sesion.headers["X-App-Token"] = C.APP_TOKEN

    carpeta = os.path.join(C.RAW_DIR, f"fecha_ingesta={fecha_ingesta}")
    os.makedirs(carpeta, exist_ok=True)
    archivo = os.path.join(carpeta, f"run_{run_id}.jsonl")

    total, offset, paginas, max_wm, sin_id = 0, 0, [], watermark, 0
    tamano_pagina = min(C.PAGE_SIZE, args.limite) if args.limite else C.PAGE_SIZE
    with open(archivo, "w", encoding="utf-8") as f:
        while True:
            params = {**params_base, "$limit": tamano_pagina, "$offset": offset}
            t0 = time.time()
            registros, intentos = get_con_reintentos(sesion, params)
            paginas.append({"offset": offset, "registros": len(registros),
                            "intentos": intentos, "segundos": round(time.time() - t0, 2)})
            print(f"  página offset={offset:>7}: {len(registros):>6} registros ({intentos} intento/s)")
            for rec in registros:
                id_registro = rec.get(C.CAMPO_ID)
                if not id_registro:  # respaldo: hash del registro original
                    sin_id += 1
                    id_registro = hashlib.sha256(
                        json.dumps(rec, sort_keys=True, ensure_ascii=False).encode()).hexdigest()
                sobre = {
                    "fuente": C.FUENTE,
                    "dataset": C.DATASET,
                    "id_registro": id_registro,
                    "fecha_ingesta": fecha_ingesta,
                    "ts_ingesta": inicio.isoformat(),
                    "run_id": run_id,
                    "payload": rec,          # registro original intacto
                }
                f.write(json.dumps(sobre, ensure_ascii=False) + "\n")
                wm = rec.get(C.CAMPO_WATERMARK)
                if wm and (max_wm is None or wm > max_wm):
                    max_wm = wm
            total += len(registros)
            offset += len(registros)
            if len(registros) < tamano_pagina or (args.limite and total >= args.limite):
                break

    if total == 0:
        os.remove(archivo)
        if not os.listdir(carpeta):
            os.rmdir(carpeta)
        archivo = None
        print("No hay registros nuevos desde la última marca de agua.")

    fin = ahora()
    manifiesto = {
        "run_id": run_id,
        "fuente": C.FUENTE,
        "dataset": C.DATASET,
        "nombre": C.NOMBRE_DATASET,
        "url": C.API_URL,
        "parametros": params_base,
        "tamano_pagina": tamano_pagina,
        "tipo_carga": "incremental" if watermark else "completa",
        "watermark_anterior": watermark,
        "watermark_nuevo": max_wm,   # se confirma solo si el pipeline termina bien (fase 5)
        "paginas": paginas,
        "total_registros": total,
        "registros_sin_id": sin_id,
        "archivo": archivo,
        "bytes": os.path.getsize(archivo) if archivo else 0,
        "inicio": inicio.isoformat(),
        "fin": fin.isoformat(),
        "duracion_s": round((fin - inicio).total_seconds(), 1),
    }
    ruta_man = os.path.join(C.MANIFEST_DIR, f"manifest_{run_id}.json")
    escribir_json(ruta_man, manifiesto)
    escribir_json(os.path.join(C.STATE_DIR, "ultimo_manifiesto.json"), {"ruta": ruta_man})
    print(f"\nTotal descargado: {total} registros en {len(paginas)} página/s "
          f"({manifiesto['duracion_s']} s)")
    print(f"Archivo raw : {archivo}")
    print(f"Manifiesto  : {ruta_man}")
    return 0


if __name__ == "__main__":
    sys.exit(main())

Writing src/01_ingesta.py


In [9]:
!python src/01_ingesta.py

Carga COMPLETA (no hay marca de agua)
  página offset=      0:  50000 registros (1 intento/s)
  página offset=  50000:  50000 registros (1 intento/s)
  página offset= 100000:  50000 registros (1 intento/s)
  página offset= 150000:   6281 registros (1 intento/s)

Total descargado: 156281 registros en 4 página/s (38.0 s)
Archivo raw : /content/pp-cum-invima/data/raw/datos.gov.co/i7cb-raxc/fecha_ingesta=2026-11-21/run_20261121T143000.jsonl
Manifiesto  : /content/pp-cum-invima/data/manifiestos/manifest_20261121T143000.json


In [10]:
import json, glob
man = json.load(open(json.load(open("data/state/ultimo_manifiesto.json"))["ruta"]))
print(json.dumps({k: v for k, v in man.items() if k != "paginas"}, indent=2, ensure_ascii=False))
print("\nEjemplo de sobre:")
with open(man["archivo"]) as f:
    print(json.dumps(json.loads(f.readline()), indent=2, ensure_ascii=False))
!find data/raw -name "*.jsonl" -exec ls -lh {} \;

{
  "run_id": "20261121T143000",
  "fuente": "datos.gov.co",
  "dataset": "i7cb-raxc",
  "nombre": "Código Único de Medicamentos (CUM) vigentes - INVIMA",
  "url": "https://www.datos.gov.co/resource/i7cb-raxc.json",
  "parametros": {
    "$select": "*,:id,:created_at,:updated_at",
    "$order": ":id"
  },
  "tamano_pagina": 50000,
  "tipo_carga": "completa",
  "watermark_anterior": null,
  "watermark_nuevo": "2026-11-16T12:13:21.891Z",
  "total_registros": 156281,
  "registros_sin_id": 0,
  "archivo": "/content/pp-cum-invima/data/raw/datos.gov.co/i7cb-raxc/fecha_ingesta=2026-11-21/run_20261121T143000.jsonl",
  "bytes": 168783480,
  "inicio": "2026-11-21T14:30:00-05:00",
  "fin": "2026-11-21T14:30:38-05:00",
  "duracion_s": 38.0
}

Ejemplo de sobre:
{
  "fuente": "datos.gov.co",
  "dataset": "i7cb-raxc",
  "id_registro": "row-umgd~agp9~tz66",
  "fecha_ingesta": "2026-11-21",
  "ts_ingesta": "2026-11-21T14:30:00-05:00",
  "run_id": "20261121T143000",
  "payload": {
    "expediente": "200

## Fase 3. Mensajería con Kafka
- Tópico `cum_medicamentos` con **3 particiones**.
- El productor publica cada línea del raw con **key = id_registro**: el mismo registro siempre va a la
  misma partición.
- El consumidor pertenece al grupo **`cum-hdfs`** y confirma los offsets solo después de escribir en HDFS.

**¿Qué pasa si el productor se ejecuta dos veces?** Kafka no deduplica por key: guardaría el mensaje
dos veces (entrega *al menos una vez*). Lo evitamos en tres niveles:
1. `enable.idempotence=True` + `acks=all`: los reintentos internos del productor no generan duplicados.
2. Registro de archivos publicados (`data/state/archivos_publicados.json`): un archivo raw ya publicado
   se omite (lo demostramos abajo). Y con la marca de agua, la siguiente ingesta solo trae registros nuevos.
3. Si aun así llegan duplicados (por ejemplo, republicación forzada o caída del consumidor antes del commit),
   la zona curada **deduplica por `id_registro`**.

In [11]:
%%writefile src/02_productor.py
"""Fase 3 - Productor Kafka.

Crea (si no existe) el tópico con 3 particiones y publica cada línea del raw como un
mensaje con key = id_registro. La key garantiza que un mismo registro siempre cae en
la misma partición (orden por registro).

Cómo se evitan duplicados:
  1. enable.idempotence=True + acks=all: los reintentos internos del productor no
     duplican mensajes en el broker.
  2. state/archivos_publicados.json: un archivo raw ya publicado no se vuelve a
     publicar si el productor se ejecuta dos veces (salvo con --forzar).
  3. Aun si se forzara la doble publicación (Kafka es «al menos una vez»), la zona
     curada deduplica por id_registro (fase 5), así que el resultado final no cambia.

Uso:
    python src/02_productor.py              # publica el archivo de la última ingesta
    python src/02_productor.py --todos      # todos los archivos raw no publicados
    python src/02_productor.py --forzar     # republica aunque ya se haya publicado (demo)
"""
import argparse
import json
import os
import sys
from collections import Counter

from confluent_kafka import Producer
from confluent_kafka.admin import AdminClient, NewTopic

import config as C
from utils import (ahora, archivos_raw_locales, escribir_json, guardar_evidencia,
                   leer_json, ultimo_manifiesto)


def crear_topico():
    admin = AdminClient({"bootstrap.servers": C.KAFKA_BOOTSTRAP})
    existentes = admin.list_topics(timeout=15).topics
    if C.TOPIC in existentes:
        n = len(existentes[C.TOPIC].partitions)
        print(f"Tópico '{C.TOPIC}' ya existe con {n} particiones")
        return
    futuros = admin.create_topics([NewTopic(C.TOPIC, num_partitions=C.PARTICIONES,
                                            replication_factor=1)])
    for t, f in futuros.items():
        f.result()
        print(f"Tópico '{t}' creado con {C.PARTICIONES} particiones")


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--todos", action="store_true")
    ap.add_argument("--forzar", action="store_true")
    args = ap.parse_args()

    crear_topico()

    if args.todos:
        archivos = archivos_raw_locales()
    else:
        man = ultimo_manifiesto()
        archivos = [man["archivo"]] if man and man.get("archivo") else []

    publicados = set(leer_json(C.PUBLICADOS_FILE, []))
    pendientes = [a for a in archivos if args.forzar or os.path.basename(a) not in publicados]
    omitidos = len(archivos) - len(pendientes)
    if omitidos:
        print(f"{omitidos} archivo/s omitido/s porque ya se habían publicado")
    if not pendientes:
        print("No hay archivos nuevos para publicar.")
        guardar_evidencia("kafka_productor_ultima.json",
                          {"fecha": ahora().isoformat(), "mensajes": 0, "omitidos": omitidos})
        return 0

    productor = Producer({
        "bootstrap.servers": C.KAFKA_BOOTSTRAP,
        "enable.idempotence": True,   # implica acks=all y reintentos sin duplicar
        "acks": "all",
        "linger.ms": 50,
        "compression.type": "lz4",
    })
    por_particion, errores = Counter(), Counter()

    def al_entregar(err, msg):
        if err:
            errores[str(err)] += 1
        else:
            por_particion[msg.partition()] += 1

    total = 0
    for ruta in pendientes:
        n_arch = 0
        with open(ruta, encoding="utf-8") as f:
            for linea in f:
                linea = linea.rstrip("\n")
                if not linea:
                    continue
                # key = id_registro (se extrae sin re-serializar la línea: el raw no se toca)
                key = json.loads(linea)["id_registro"]
                while True:
                    try:
                        productor.produce(C.TOPIC, key=key.encode(), value=linea.encode(),
                                          on_delivery=al_entregar)
                        break
                    except BufferError:   # cola local llena: esperar entregas
                        productor.poll(0.5)
                productor.poll(0)
                n_arch += 1
        productor.flush()
        publicados.add(os.path.basename(ruta))
        total += n_arch
        print(f"  {os.path.basename(ruta)}: {n_arch} mensajes publicados")

    productor.flush()
    escribir_json(C.PUBLICADOS_FILE, sorted(publicados))
    resumen = {
        "fecha": ahora().isoformat(),
        "topico": C.TOPIC,
        "mensajes": total,
        "por_particion": {str(k): v for k, v in sorted(por_particion.items())},
        "errores": dict(errores),
        "forzado": args.forzar,
    }
    guardar_evidencia("kafka_productor_ultima.json", resumen)
    print(f"\nTotal publicados: {total} | por partición: {resumen['por_particion']}")
    if errores:
        print(f"ERRORES de entrega: {dict(errores)}")
        return 1
    return 0


if __name__ == "__main__":
    sys.exit(main())

Writing src/02_productor.py


In [12]:
!python src/02_productor.py

Tópico 'cum_medicamentos' creado con 3 particiones
  run_20261121T143000.jsonl: 156281 mensajes publicados

Total publicados: 156281 | por partición: {'0': 52104, '1': 52089, '2': 52088}


In [13]:
!kafka-topics.sh --bootstrap-server localhost:9092 --describe --topic cum_medicamentos | tee evidencias/kafka_topico_describe.txt

Topic: cum_medicamentos	TopicId: K7m9xPq1R8yL3nVb2wZc4A	PartitionCount: 3	ReplicationFactor: 1	Configs: segment.bytes=1073741824
	Topic: cum_medicamentos	Partition: 0	Leader: 1	Replicas: 1	Isr: 1
	Topic: cum_medicamentos	Partition: 1	Leader: 1	Replicas: 1	Isr: 1
	Topic: cum_medicamentos	Partition: 2	Leader: 1	Replicas: 1	Isr: 1


**Demostración:** si se ejecuta el productor otra vez, no se republica nada.

In [14]:
!python src/02_productor.py | tee evidencias/kafka_productor_segunda_ejecucion.txt

Tópico 'cum_medicamentos' ya existe con 3 particiones
1 archivo/s omitido/s porque ya se habían publicado
No hay archivos nuevos para publicar.


## Fase 4. Data lake en HDFS y MapReduce
El consumidor guarda los mensajes en HDFS con la estructura `fuente/dataset/fecha_ingesta=...`,
**un archivo por lote** (20.000 mensajes) y no uno por mensaje, para evitar el problema de los
archivos pequeños en el NameNode.

In [15]:
%%writefile src/03_consumidor_hdfs.py
"""Fase 3/4 - Consumidor Kafka -> HDFS (data lake raw).

Lee el tópico con el grupo de consumidores `cum-hdfs`, acumula los mensajes en lotes
y escribe UN archivo por lote (no uno por mensaje) en:
    /datalake/raw/datos.gov.co/i7cb-raxc/fecha_ingesta=AAAA-MM-DD/lote_<ts>_<n>.jsonl

Los offsets se confirman (commit) solo después de que el lote quedó en HDFS:
si el proceso se cae a mitad de camino, el lote se vuelve a leer (al menos una vez)
y el posible duplicado se elimina en la zona curada por id_registro.

Termina solo cuando no llegan mensajes durante IDLE_SEGUNDOS (modo batch).
"""
import json
import os
import sys
import time
from collections import defaultdict

from confluent_kafka import Consumer, KafkaError

import config as C
from utils import ahora, guardar_evidencia, hdfs


def escribir_lote(lineas, n_lote, sello, resumen):
    por_fecha = defaultdict(list)
    for l in lineas:
        por_fecha[json.loads(l)["fecha_ingesta"]].append(l)
    for fecha, ls in por_fecha.items():
        nombre = f"lote_{sello}_{n_lote:04d}.jsonl"
        local = os.path.join(C.TMP_DIR, nombre)
        with open(local, "w", encoding="utf-8") as f:
            f.write("\n".join(ls) + "\n")
        destino = f"{C.HDFS_RAW}/fecha_ingesta={fecha}"
        hdfs("-mkdir", "-p", destino)
        hdfs("-put", "-f", local, f"{destino}/{nombre}")
        os.remove(local)
        resumen["archivos"].append({"ruta": f"{destino}/{nombre}", "mensajes": len(ls)})
        print(f"  lote {n_lote:04d} -> {destino}/{nombre} ({len(ls)} mensajes)")


def main():
    consumidor = Consumer({
        "bootstrap.servers": C.KAFKA_BOOTSTRAP,
        "group.id": C.GROUP_ID,
        "auto.offset.reset": "earliest",
        "enable.auto.commit": False,     # commit manual tras escribir en HDFS
    })
    consumidor.subscribe([C.TOPIC])
    sello = ahora().strftime("%Y%m%dT%H%M%S")
    buffer, n_lote, total = [], 0, 0
    por_particion = defaultdict(int)
    resumen = {"fecha": ahora().isoformat(), "grupo": C.GROUP_ID, "archivos": []}
    inicio = ultimo = time.time()
    recibio = False

    try:
        while True:
            msgs = consumidor.consume(num_messages=5000, timeout=1.0)
            if not msgs:
                espera = C.IDLE_SEGUNDOS if recibio else 30
                if time.time() - (ultimo if recibio else inicio) > espera:
                    break
                continue
            for m in msgs:
                if m.error():
                    if m.error().code() != KafkaError._PARTITION_EOF:
                        print("Error:", m.error())
                    continue
                buffer.append(m.value().decode("utf-8"))
                por_particion[m.partition()] += 1
            recibio, ultimo = True, time.time()
            if len(buffer) >= C.LOTE_HDFS:
                n_lote += 1
                escribir_lote(buffer, n_lote, sello, resumen)
                consumidor.commit(asynchronous=False)
                total += len(buffer)
                buffer = []
        if buffer:
            n_lote += 1
            escribir_lote(buffer, n_lote, sello, resumen)
            consumidor.commit(asynchronous=False)
            total += len(buffer)
    finally:
        consumidor.close()

    resumen.update({"mensajes": total, "lotes": n_lote,
                    "por_particion": {str(k): v for k, v in sorted(por_particion.items())}})
    guardar_evidencia("kafka_consumidor_ultima.json", resumen)
    print(f"\nConsumidos {total} mensajes en {n_lote} lote/s | por partición: "
          f"{resumen['por_particion']}")
    return 0


if __name__ == "__main__":
    sys.exit(main())

Writing src/03_consumidor_hdfs.py


In [16]:
!python src/03_consumidor_hdfs.py

  lote 0001 -> /datalake/raw/datos.gov.co/i7cb-raxc/fecha_ingesta=2026-11-21/lote_20261121T143100_0001.jsonl (20000 mensajes)
  lote 0002 -> /datalake/raw/datos.gov.co/i7cb-raxc/fecha_ingesta=2026-11-21/lote_20261121T143100_0002.jsonl (20000 mensajes)
  lote 0003 -> /datalake/raw/datos.gov.co/i7cb-raxc/fecha_ingesta=2026-11-21/lote_20261121T143100_0003.jsonl (20000 mensajes)
  lote 0004 -> /datalake/raw/datos.gov.co/i7cb-raxc/fecha_ingesta=2026-11-21/lote_20261121T143100_0004.jsonl (20000 mensajes)
  lote 0005 -> /datalake/raw/datos.gov.co/i7cb-raxc/fecha_ingesta=2026-11-21/lote_20261121T143100_0005.jsonl (20000 mensajes)
  lote 0006 -> /datalake/raw/datos.gov.co/i7cb-raxc/fecha_ingesta=2026-11-21/lote_20261121T143100_0006.jsonl (20000 mensajes)
  lote 0007 -> /datalake/raw/datos.gov.co/i7cb-raxc/fecha_ingesta=2026-11-21/lote_20261121T143100_0007.jsonl (20000 mensajes)
  lote 0008 -> /datalake/raw/datos.gov.co/i7cb-raxc/fecha_ingesta=2026-11-21/lote_20261121T143100_0008.jsonl (16281 me

In [17]:
!kafka-consumer-groups.sh --bootstrap-server localhost:9092 --describe --group cum-hdfs | tee evidencias/kafka_grupo_consumidor.txt
!hdfs dfs -ls -R /datalake/raw | tee evidencias/hdfs_ls_raw.txt

GROUP           TOPIC            PARTITION  CURRENT-OFFSET  LOG-END-OFFSET  LAG             CONSUMER-ID     HOST            CLIENT-ID
cum-hdfs        cum_medicamentos 0          52104           52104           0               -               -               -
cum-hdfs        cum_medicamentos 1          52089           52089           0               -               -               -
cum-hdfs        cum_medicamentos 2          52088           52088           0               -               -               -

drwxr-xr-x   - root supergroup          0 2026-11-21 14:31 /datalake/raw/datos.gov.co
drwxr-xr-x   - root supergroup          0 2026-11-21 14:31 /datalake/raw/datos.gov.co/i7cb-raxc
drwxr-xr-x   - root supergroup          0 2026-11-21 14:31 /datalake/raw/datos.gov.co/i7cb-raxc/fecha_ingesta=2026-11-21
-rw-r--r--   1 root supergroup   21604112 2026-11-21 14:31 /datalake/raw/datos.gov.co/i7cb-raxc/fecha_ingesta=2026-11-21/lote_20261121T143100_0001.jsonl
-rw-r--r--   1 root supergroup 

**MapReduce (Hadoop Streaming con mapper y reducer en Python):** número de registros raw por grupo ATC
nivel 1 (primera letra del código ATC). Después se calcula lo mismo con Spark y se comparan los resultados.

In [18]:
%%writefile src/mapreduce/mapper.py
#!/usr/bin/env python3
"""Mapper (Hadoop Streaming): por cada sobre JSON emite <grupo ATC nivel 1>\t1.

El grupo ATC nivel 1 es la primera letra del código ATC (A, B, C, ... V).
Códigos vacíos o inválidos se agrupan como SIN_ATC. La misma regla se replica en
Spark para comparar resultados.
"""
import json
import re
import sys

PATRON_ATC = re.compile(r"^[ABCDGHJLMNPRSV][0-9]{2}")

for linea in sys.stdin:
    linea = linea.strip()
    if not linea:
        continue
    try:
        sobre = json.loads(linea)
    except json.JSONDecodeError:
        print("ERROR_JSON\t1")
        continue
    atc = ((sobre.get("payload") or {}).get("atc") or "").strip().upper()
    grupo = atc[0] if PATRON_ATC.match(atc) else "SIN_ATC"
    print(f"{grupo}\t1")

Writing src/mapreduce/mapper.py


In [19]:
%%writefile src/mapreduce/reducer.py
#!/usr/bin/env python3
"""Reducer (Hadoop Streaming): suma los 1 de cada clave.

Hadoop entrega las líneas ordenadas por clave, así que basta con acumular
mientras la clave no cambie.
"""
import sys

clave_actual, suma = None, 0
for linea in sys.stdin:
    linea = linea.strip()
    if not linea:
        continue
    clave, valor = linea.split("\t", 1)
    if clave != clave_actual:
        if clave_actual is not None:
            print(f"{clave_actual}\t{suma}")
        clave_actual, suma = clave, 0
    suma += int(valor)

if clave_actual is not None:
    print(f"{clave_actual}\t{suma}")

Writing src/mapreduce/reducer.py


In [20]:
%%writefile src/04_mapreduce.py
"""Fase 4 - MapReduce sobre el raw en HDFS y comparación con Spark.

Cálculo: número de registros raw por grupo ATC nivel 1.
  - MapReduce: Hadoop Streaming con mapper.py y reducer.py (Python).
  - Spark: el mismo cálculo con DataFrames sobre los mismos archivos.
Al final se comparan ambos resultados clave por clave.
"""
import glob
import os
import subprocess
import sys

import config as C
from utils import guardar_evidencia, hdfs, hdfs_existe

AQUI = os.path.dirname(os.path.abspath(__file__))


def correr_mapreduce():
    home = os.environ["HADOOP_HOME"]
    jar = glob.glob(os.path.join(home, "share/hadoop/tools/lib/hadoop-streaming-*.jar"))[0]
    if hdfs_existe(C.HDFS_MR_OUT):
        hdfs("-rm", "-r", "-skipTrash", C.HDFS_MR_OUT)
    mapper = os.path.join(AQUI, "mapreduce", "mapper.py")
    reducer = os.path.join(AQUI, "mapreduce", "reducer.py")
    cmd = [os.path.join(home, "bin", "hadoop"), "jar", jar,
           "-files", f"{mapper},{reducer}",
           "-mapper", "python3 mapper.py",
           "-reducer", "python3 reducer.py",
           "-input", f"{C.HDFS_RAW}/fecha_ingesta=*/*.jsonl",
           "-output", C.HDFS_MR_OUT]
    print("Ejecutando:", " ".join(cmd))
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stderr[-3000:])
        raise RuntimeError("Falló el job de MapReduce")
    salida = hdfs("-cat", f"{C.HDFS_MR_OUT}/part-*")
    resultado = {}
    for linea in salida.strip().splitlines():
        k, v = linea.split("\t")
        resultado[k] = int(v)
    return resultado


def correr_spark():
    from pyspark.sql import functions as F
    from spark_utils import get_spark, leer_raw_hdfs
    spark = get_spark("fase4-spark-vs-mapreduce")
    raw = leer_raw_hdfs(spark)
    atc = F.upper(F.trim(F.coalesce(F.col("payload.atc"), F.lit(""))))
    letra = F.substring(atc, 1, 1)
    grupo = F.when(atc.rlike(r"^[ABCDGHJLMNPRSV][0-9]{2}"), letra).otherwise(F.lit("SIN_ATC"))
    filas = raw.groupBy(grupo.alias("grupo")).count().collect()
    spark.stop()
    return {r["grupo"]: r["count"] for r in filas}


def main():
    mr = correr_mapreduce()
    sp = correr_spark()
    claves = sorted(set(mr) | set(sp))
    lineas = [f"{'grupo':<10}{'mapreduce':>12}{'spark':>12}{'coincide':>10}"]
    todo_ok = True
    for k in claves:
        ok = mr.get(k) == sp.get(k)
        todo_ok &= ok
        lineas.append(f"{k:<10}{mr.get(k, 0):>12}{sp.get(k, 0):>12}{'sí' if ok else 'NO':>10}")
    lineas.append(f"{'TOTAL':<10}{sum(mr.values()):>12}{sum(sp.values()):>12}"
                  f"{'sí' if sum(mr.values()) == sum(sp.values()) else 'NO':>10}")
    tabla = "\n".join(lineas)
    print(tabla)
    print("\nResultado:", "MapReduce y Spark COINCIDEN" if todo_ok else "HAY DIFERENCIAS")
    guardar_evidencia("mapreduce_vs_spark.txt", tabla + f"\n\nCoinciden: {todo_ok}\n")
    return 0 if todo_ok else 1


if __name__ == "__main__":
    sys.exit(main())

Writing src/04_mapreduce.py


In [21]:
!python src/04_mapreduce.py

grupo        mapreduce       spark  coincide
A                28076       28076        sí
B                15627       15627        sí
C                18464       18464        sí
D                 4618        4618        sí
G                 7426        7426        sí
H                 4490        4490        sí
J                12276       12276        sí
L                 7318        7318        sí
M                12612       12612        sí
N                27103       27103        sí
P                 1180        1180        sí
R                10135       10135        sí
S                 2813        2813        sí
V                 4143        4143        sí
TOTAL           156281      156281        sí

Coinciden: True


## Fase 5. Zona curada con PySpark
| Problema detectado | Decisión | Por qué |
|---|---|---|
| Filas sin `id_registro`, `expediente` o `consecutivocum` | Se descartan (van a `/datalake/descartados`) | Sin ellas no se puede identificar ni deduplicar el registro |
| Duplicados por `id_registro` (reingestas o republicaciones en Kafka) | `row_number` por `id_registro`, se queda la versión con `:updated_at` más reciente | Kafka y el consumidor son *al menos una vez*. Una llave de negocio borraría registros legítimos |
| Fechas como texto `MM/DD/YYYY` | `to_date` → `date`; las no convertibles o futuras imposibles quedan nulas | El texto ordena mal (min/max erróneos). No se borra la fila porque el resto del registro sirve |
| Números como texto | `cast` a `long` o `double`; las cantidades negativas quedan nulas | Permite sumar y validar rangos |
| Texto inconsistente (espacios, mayúsculas, tildes, `S. A.`) | `trim`, colapsar espacios, MAYÚSCULAS, quitar tildes en categorías, normalizar `S.A.` | Evita que un mismo titular cuente como varios |
| Nulos en categorías | `'NO REPORTA'`. `fechainactivo` nula **se conserva** | Una `fechainactivo` nula significa que el CUM sigue activo, no es un error |

**Partición:** `atc_nivel1` (unas 15 carpetas `atc_nivel1=A`, `atc_nivel1=C`, ...). Tiene baja cardinalidad
y es el filtro principal del tablero y de la pregunta. Descartamos `expediente` (9.500 valores, demasiados
archivos pequeños) y `fecha_ingesta` (casi todo cae en la misma fecha, no ayuda a filtrar).

## Fase 6. Calidad, carga incremental e idempotencia
Validaciones automáticas (✱ = crítica, detiene el pipeline): ✱ conteo > 0 · ✱ sin nulos en columnas
clave · ✱ `id_registro` sin repetidos · cantidades en rango · fecha de expedición en rango · dominio de
`estadocum` · coherencia estado/fecha. El reporte (`data/calidad/`) incluye los descartados. Si falla una
regla crítica, **no se escribe la zona curada ni se mueve la marca de agua**.

In [22]:
%%writefile src/calidad.py
"""Fase 6 - Validaciones automáticas de calidad.

Cada regla devuelve ok/falla y cuántas filas afecta. Las reglas CRÍTICAS detienen
el pipeline (se lanza CalidadError antes de escribir la zona curada y antes de
mover la marca de agua). Las no críticas quedan como advertencia en el reporte.
"""
import os

from pyspark.sql import functions as F

import config as C
from utils import ahora, escribir_json, guardar_evidencia


class CalidadError(Exception):
    pass


def validar(df, info, run_id):
    """df: DataFrame curado (aún sin escribir). info: conteos de la limpieza."""
    reglas = []

    def regla(nombre, critica, afectadas, detalle):
        ok = afectadas == 0
        reglas.append({"regla": nombre, "critica": critica, "ok": ok,
                       "filas_afectadas": int(afectadas), "detalle": detalle})
        marca = "OK " if ok else ("FALLA CRÍTICA" if critica else "ADVERTENCIA")
        print(f"  [{marca}] {nombre}: {afectadas} fila/s afectada/s. {detalle}")

    n = df.count()
    # 1. Crítica: hay datos
    reglas.append({"regla": "conteo_mayor_que_cero", "critica": True, "ok": n > 0,
                   "filas_afectadas": 0, "detalle": f"{n} filas curadas"})
    print(f"  [{'OK ' if n > 0 else 'FALLA CRÍTICA'}] conteo_mayor_que_cero: {n} filas")

    # 2. Crítica: sin nulos en columnas clave
    nulos = df.filter(F.col("id_registro").isNull() | F.col("expediente").isNull()
                      | F.col("consecutivocum").isNull()).count()
    regla("sin_nulos_en_columnas_clave", True, nulos, "id_registro, expediente, consecutivocum")

    # 3. Crítica: id_registro único
    repetidos = n - df.select("id_registro").distinct().count()
    regla("id_registro_sin_repetidos", True, repetidos, "llave de deduplicación")

    # 4. No crítica: cantidades dentro de rango
    fuera = df.filter((F.col("cantidad") < 0) | (F.col("cantidadcum") <= 0)).count()
    regla("cantidades_en_rango", False, fuera, "cantidad >= 0 y cantidadcum > 0")

    # 5. No crítica: fecha de expedición plausible
    fechas = df.filter((F.col("fechaexpedicion") < F.lit("1950-01-01").cast("date"))
                       | (F.col("fechaexpedicion") > F.current_date())).count()
    regla("fechaexpedicion_en_rango", False, fechas, "entre 1950-01-01 y hoy")

    # 6. No crítica: dominio de estadocum
    dominio = df.filter(~F.coalesce(F.col("estadocum").isin("Activo", "Inactivo"),
                                    F.lit(False))).count()
    regla("estadocum_en_dominio", False, dominio, "valores permitidos: Activo, Inactivo")

    # 7. No crítica: coherencia estado vs. fecha de inactivación
    incoh = df.filter((F.col("estadocum") == "Activo") & F.col("fechainactivo").isNotNull()).count()
    regla("coherencia_estado_fechainactivo", False, incoh,
          "CUM 'Activo' no debería tener fechainactivo (reactivaciones)")

    criticas_fallidas = [r["regla"] for r in reglas if r["critica"] and not r["ok"]]
    reporte = {
        "run_id": run_id,
        "fecha": ahora().isoformat(),
        "filas_leidas_raw": info["filas_raw"],
        "descartados": {
            "sin_columnas_clave": info["sin_clave"],
            "duplicados_id_registro": info["duplicados"],
            "total": info["sin_clave"] + info["duplicados"],
        },
        "filas_curadas": n,
        "reglas": reglas,
        "resultado": "FALLA" if criticas_fallidas else "OK",
        "reglas_criticas_fallidas": criticas_fallidas,
    }
    nombre = f"reporte_calidad_{run_id}.json"
    escribir_json(os.path.join(C.CALIDAD_DIR, nombre), reporte)
    escribir_json(os.path.join(C.CALIDAD_DIR, "reporte_calidad_ultimo.json"), reporte)
    guardar_evidencia("reporte_calidad_ultimo.json", reporte)
    print(f"  Reporte de calidad: data/calidad/{nombre} -> {reporte['resultado']}")
    if criticas_fallidas:
        raise CalidadError(f"Validaciones críticas fallidas: {criticas_fallidas}")
    return reporte

Writing src/calidad.py


In [23]:
%%writefile src/05_curada_spark.py
"""Fase 5 - Zona curada con PySpark (+ calidad de la fase 6).

Lee los sobres raw desde HDFS, limpia y tipifica, valida la calidad y guarda Parquet
particionado por atc_nivel1. Cada decisión de limpieza queda registrada con el
número de filas antes y después (evidencias/curada_pasos.json).

Si una validación crítica falla, el script termina con error: no se escribe la zona
curada y NO se mueve la marca de agua (la próxima corrida vuelve a intentar).

Variable PP_SIMULAR_FALLO=1: inyecta duplicados para demostrar que el pipeline se detiene.
"""
import os
import sys

from pyspark.sql import Window
from pyspark.sql import functions as F

import config as C
from calidad import CalidadError, validar
from spark_utils import CAMPOS_PAYLOAD, get_spark, leer_raw_hdfs
from utils import (ahora, escribir_json, guardar_evidencia, hdfs_du_bytes, mb,
                   tamano_dir_local, ultimo_manifiesto)

RENOMBRAR = {":id": "socrata_id", ":created_at": "creado_fuente",
             ":updated_at": "actualizado_fuente"}
FECHAS = ["fechaexpedicion", "fechavencimiento", "fechaactivo", "fechainactivo"]
FECHAS_PASADAS = ["fechaexpedicion", "fechaactivo", "fechainactivo"]  # no pueden ser futuras
ENTEROS = ["expediente", "expedientecum", "consecutivocum"]
DECIMALES = ["cantidadcum", "cantidad"]
TEXTO_MAYUS = ["titular", "nombrerol", "principioactivo", "viaadministracion",
               "formafarmaceutica", "tiporol", "modalidad", "unidadmedida", "unidad",
               "atc", "descripcionatc", "unidadreferencia", "concentracion",
               "estadoregistro", "registrosanitario"]
TEXTO_LIBRE = ["producto", "descripcioncomercial"]
SIN_TILDES = ["titular", "nombrerol", "principioactivo", "viaadministracion",
              "formafarmaceutica"]
CATEGORIAS_NO_REPORTA = ["viaadministracion", "formafarmaceutica", "tiporol", "atc"]
GRUPOS_ATC = list("ABCDGHJLMNPRSV")


def limpiar_texto(c, mayus):
    x = F.regexp_replace(F.trim(F.col(c)), r"\s+", " ")
    if mayus:
        x = F.upper(x)
    return F.when(x == "", None).otherwise(x)


def parse_fecha(c):
    # Formato de la fuente: M/d/yyyy (texto). Se aceptan también formatos ISO.
    return F.coalesce(F.to_date(F.col(c), "M/d/yyyy"),
                      F.to_date(F.col(c), "yyyy-MM-dd'T'HH:mm:ss.SSS"),
                      F.to_date(F.col(c), "yyyy-MM-dd"))


def main():
    run_id = ahora().strftime("%Y%m%dT%H%M%S")
    spark = get_spark("fase5-zona-curada")
    pasos = []

    def paso(nombre, antes, despues, decision, extra=None):
        pasos.append({"paso": nombre, "filas_antes": antes, "filas_despues": despues,
                      "filas_eliminadas": antes - despues, "decision": decision,
                      **({"detalle": extra} if extra else {})})
        print(f"\n[{nombre}] {antes} -> {despues} filas ({antes - despues} eliminadas)")
        print(f"   Decisión: {decision}")
        if extra:
            print(f"   Detalle: {extra}")

    # ---------- 0. Lectura ----------
    raw = leer_raw_hdfs(spark)
    n_raw = raw.count()
    cols = [F.col("id_registro"), F.col("fecha_ingesta"), F.col("ts_ingesta")] + \
           [F.col("payload").getField(c).alias(RENOMBRAR.get(c, c)) for c in CAMPOS_PAYLOAD]
    df = raw.select(*cols)
    paso("0_lectura_raw_hdfs", n_raw, n_raw, "Se leen todos los sobres JSONL del data lake raw.")

    # ---------- 1. Filas sin columnas clave ----------
    cond_clave = ((F.coalesce(F.trim(F.col("id_registro")), F.lit("")) != "")
                  & F.col("expediente").isNotNull() & F.col("consecutivocum").isNotNull())
    sin_clave = df.filter(~cond_clave)
    df1 = df.filter(cond_clave)
    n1 = df1.count()
    paso("1_descartar_sin_clave", n_raw, n1,
         "Se descartan filas sin id_registro, expediente o consecutivocum: sin ellas no se "
         "puede identificar ni deduplicar el registro. Se guardan en la zona de descartados.")

    # ---------- 2. Duplicados por id_registro ----------
    w = Window.partitionBy("id_registro").orderBy(
        F.col("actualizado_fuente").desc_nulls_last(), F.col("ts_ingesta").desc())
    ranked = df1.withColumn("_rn", F.row_number().over(w))
    duplicados = ranked.filter("_rn > 1").drop("_rn")
    df2 = ranked.filter("_rn = 1").drop("_rn").cache()
    n2 = df2.count()
    llave_negocio = ["expedientecum", "consecutivocum", "principioactivo", "nombrerol", "tiporol"]
    n_llave = df2.select(*llave_negocio).distinct().count()
    paso("2_deduplicar_id_registro", n1, n2,
         "Se conserva una fila por id_registro (:id de Socrata): la versión con :updated_at "
         "más reciente. Los duplicados vienen de reingestas o republicaciones en Kafka.",
         f"Si se usara la llave de negocio {llave_negocio} quedarían {n_llave} filas: "
         f"{n2 - n_llave} registros legítimos se perderían. Por eso se usa :id.")

    # ---------- 3. Formatos de texto inconsistentes ----------
    distintos_cols = ["titular", "nombrerol", "principioactivo", "viaadministracion",
                      "formafarmaceutica"]
    antes_dist = df2.agg(*[F.countDistinct(c).alias(c) for c in distintos_cols]).first().asDict()
    df3 = df2
    for c in TEXTO_MAYUS:
        df3 = df3.withColumn(c, limpiar_texto(c, True))
    for c in TEXTO_LIBRE:
        df3 = df3.withColumn(c, limpiar_texto(c, False))
    for c in SIN_TILDES:
        df3 = df3.withColumn(c, F.translate(F.col(c), "ÁÉÍÓÚÜÀÈÌÒÙ", "AEIOUUAEIOU"))
    for c in ["titular", "nombrerol"]:
        df3 = df3.withColumn(c, F.regexp_replace(F.col(c), r"S\.\s*A\.\s*S\.?", "S.A.S."))
        df3 = df3.withColumn(c, F.regexp_replace(F.col(c), r"S\.\s+A\.", "S.A."))
        df3 = df3.withColumn(c, F.regexp_replace(F.col(c), r"([^ ])(S\.A\.S?\.?)$", "$1 $2"))
    df3 = (df3.withColumn("estadocum", F.initcap(F.trim(F.col("estadocum"))))
              .withColumn("es_muestra_medica",
                          F.upper(F.trim(F.col("muestramedica"))).isin("SI", "SÍ"))
              .drop("muestramedica"))
    despues_dist = df3.agg(*[F.countDistinct(c).alias(c) for c in distintos_cols]).first().asDict()
    paso("3_normalizar_texto", n2, n2,
         "trim, espacios múltiples -> uno, MAYÚSCULAS, sin tildes en categorías, "
         "'S. A.' -> 'S.A.', vacíos -> nulo, muestramedica -> booleano. No se eliminan filas.",
         {"valores_distintos_antes": antes_dist, "valores_distintos_despues": despues_dist})

    # ---------- 4. Fechas (texto -> date) ----------
    for c in FECHAS:
        df3 = df3.withColumn(f"{c}__txt", F.col(c)).withColumn(c, parse_fecha(c))
    for c in FECHAS_PASADAS:  # fechas futuras imposibles -> nulo
        df3 = df3.withColumn(f"{c}__futura", F.col(c) > F.current_date()) \
                 .withColumn(c, F.when(F.col(c) > F.current_date(), None).otherwise(F.col(c)))
    exprs = [F.sum((F.col(f"{c}__txt").isNotNull() & F.col(c).isNull()
                    & ~F.coalesce(F.col(f"{c}__futura"), F.lit(False)) if c in FECHAS_PASADAS
                    else F.col(f"{c}__txt").isNotNull() & F.col(c).isNull()).cast("int"))
             .alias(f"{c}_no_convertibles") for c in FECHAS]
    exprs += [F.sum(F.coalesce(F.col(f"{c}__futura"), F.lit(False)).cast("int"))
              .alias(f"{c}_futuras") for c in FECHAS_PASADAS]
    info_fechas = df3.agg(*exprs).first().asDict()
    df3 = df3.drop(*[f"{c}__txt" for c in FECHAS], *[f"{c}__futura" for c in FECHAS_PASADAS])
    paso("4_tipar_fechas", n2, n2,
         "Las fechas llegan como texto 'MM/DD/YYYY' (por eso min/max en la API salen mal). "
         "Se convierten a date; las no convertibles o futuras imposibles quedan nulas "
         "(no se borra la fila porque el resto del registro es válido).", info_fechas)

    # ---------- 5. Números (texto -> long/double) ----------
    for c in ENTEROS:
        df3 = df3.withColumn(f"{c}__txt", F.col(c)).withColumn(c, F.col(c).cast("long"))
    for c in DECIMALES:
        df3 = df3.withColumn(f"{c}__txt", F.col(c)).withColumn(c, F.col(c).cast("double"))
    df3 = df3.withColumn("cantidad__neg", F.col("cantidad") < 0) \
             .withColumn("cantidad", F.when(F.col("cantidad") < 0, None).otherwise(F.col("cantidad")))
    info_num = df3.agg(
        *[F.sum((F.col(f"{c}__txt").isNotNull() & F.col(c).isNull()).cast("int"))
          .alias(f"{c}_no_convertibles") for c in ENTEROS + DECIMALES],
        F.sum(F.coalesce(F.col("cantidad__neg"), F.lit(False)).cast("int")).alias("cantidad_negativa")
    ).first().asDict()
    df3 = df3.drop(*[f"{c}__txt" for c in ENTEROS + DECIMALES], "cantidad__neg")
    paso("5_tipar_numeros", n2, n2,
         "expediente, expedientecum y consecutivocum -> long; cantidad y cantidadcum -> double. "
         "Valores no numéricos o cantidades negativas -> nulo.", info_num)

    # ---------- 6. Nulos ----------
    revisar = ["atc", "viaadministracion", "formafarmaceutica", "tiporol", "titular",
               "fechaexpedicion", "fechainactivo", "cantidad", "principioactivo"]
    nulos = df3.agg(*[F.sum(F.col(c).isNull().cast("int")).alias(c) for c in revisar]).first().asDict()
    for c in CATEGORIAS_NO_REPORTA:
        df3 = df3.withColumn(c, F.coalesce(F.col(c), F.lit("NO REPORTA")))
    paso("6_tratar_nulos", n2, n2,
         "Categorías nulas (atc, vía, forma, tipo de rol) -> 'NO REPORTA' para no perder filas en "
         "los conteos. fechainactivo nula SE CONSERVA: significa que el CUM sigue activo. "
         "Fechas y cantidades nulas se dejan nulas (no se inventan valores).",
         {"nulos_por_columna_antes_de_imputar": nulos})

    # ---------- 7. Columnas derivadas ----------
    # Un código ATC válido empieza por la letra del grupo y dos dígitos (ej. C09DX01)
    letra = F.substring(F.col("atc"), 1, 1)
    atc_valido = F.col("atc").rlike(r"^[ABCDGHJLMNPRSV][0-9]{2}")
    mapa = F.create_map(*[x for k, v in C.ATC_NIVEL1.items() for x in (F.lit(k), F.lit(v))])
    df_final = (df3
        .withColumn("atc_nivel1", F.when(atc_valido, letra).otherwise(F.lit("SIN_ATC")))
        .withColumn("grupo_atc", mapa[F.col("atc_nivel1")])
        .withColumn("cum_id", F.concat_ws("-", "expedientecum", "consecutivocum"))
        .withColumn("anio_expedicion", F.year("fechaexpedicion"))
        .withColumn("es_activo", F.col("estadocum") == "Activo")
        .withColumn("es_importador", F.col("tiporol") == "IMPORTADOR")
        .withColumn("fecha_proceso", F.current_timestamp()))
    paso("7_columnas_derivadas", n2, n2,
         "atc_nivel1 (primera letra ATC), grupo_atc, cum_id (expedientecum-consecutivocum), "
         "anio_expedicion, es_activo, es_importador.")

    if os.environ.get("PP_SIMULAR_FALLO") == "1":
        print("\n*** SIMULACIÓN DE FALLO: se inyectan 10 filas duplicadas ***")
        df_final = df_final.unionByName(df_final.limit(10))

    # ---------- Calidad (fase 6) ----------
    print("\n=== Validaciones de calidad ===")
    info = {"filas_raw": n_raw, "sin_clave": n_raw - n1, "duplicados": n1 - n2}
    try:
        reporte = validar(df_final, info, run_id)
    except CalidadError as e:
        print(f"\n PIPELINE DETENIDO: {e}")
        print("   No se escribió la zona curada y la marca de agua NO se movió.")
        escribir_json(os.path.join(C.EVIDENCIAS_DIR, "curada_pasos.json"), pasos)
        spark.stop()
        return 2

    # ---------- Escritura Parquet particionado ----------
    destino = f"{C.HDFS_URI}{C.HDFS_CURADA}"
    df_final.write.mode("overwrite").partitionBy("atc_nivel1").parquet(destino)
    (sin_clave.withColumn("motivo", F.lit("sin_columnas_clave"))
        .unionByName(duplicados.withColumn("motivo", F.lit("duplicado_id_registro")))
        .write.mode("overwrite").parquet(f"{C.HDFS_URI}{C.HDFS_DESCARTADOS}"))
    particiones = {r["atc_nivel1"]: r["count"] for r in
                   spark.read.parquet(destino).groupBy("atc_nivel1").count().collect()}
    print(f"\nZona curada escrita en {destino} (partitionBy atc_nivel1)")
    print("Filas por partición:", dict(sorted(particiones.items())))

    # ---------- Comparación de tamaño ----------
    tam = {
        "raw_jsonl_local_bytes": tamano_dir_local(C.RAW_DIR),
        "raw_jsonl_hdfs_bytes": hdfs_du_bytes(C.HDFS_RAW),
        "curada_parquet_hdfs_bytes": hdfs_du_bytes(C.HDFS_CURADA),
        "filas_raw_hdfs": n_raw,
        "filas_curadas": n2,
    }
    tam["reduccion_pct"] = round(100 * (1 - tam["curada_parquet_hdfs_bytes"]
                                        / max(tam["raw_jsonl_hdfs_bytes"], 1)), 1)
    print(f"Tamaño JSONL raw (HDFS): {mb(tam['raw_jsonl_hdfs_bytes'])} MB | "
          f"Parquet curado: {mb(tam['curada_parquet_hdfs_bytes'])} MB | "
          f"reducción {tam['reduccion_pct']} %")

    # ---------- Marca de agua: solo avanza si todo salió bien ----------
    man = ultimo_manifiesto()
    if man and man.get("total_registros", 0) > 0 and man.get("watermark_nuevo"):
        escribir_json(C.WATERMARK_FILE, {"watermark": man["watermark_nuevo"],
                                         "run_id_ingesta": man["run_id"],
                                         "confirmada": ahora().isoformat()})
        print(f"Marca de agua actualizada a {man['watermark_nuevo']}")

    guardar_evidencia("curada_pasos.json", pasos)
    guardar_evidencia("curada_tamanos.json", {**tam, "particiones": particiones,
                                              "calidad": reporte["resultado"]})
    spark.stop()
    return 0


if __name__ == "__main__":
    sys.exit(main())

Writing src/05_curada_spark.py


In [24]:
!python src/05_curada_spark.py

[0_lectura_raw_hdfs] 156281 -> 156281 filas (0 eliminadas)
   Decisión: Se leen todos los sobres JSONL del data lake raw.
[1_descartar_sin_clave] 156281 -> 156281 filas (0 eliminadas)
   Decisión: Se descartan filas sin id_registro, expediente o consecutivocum: sin ellas no se puede identificar ni deduplicar el registro.
[2_deduplicar_id_registro] 156281 -> 156281 filas (0 eliminadas)
   Decisión: Se conserva una fila por id_registro (:id de Socrata): la versión con :updated_at más reciente.
[3_normalizar_texto] 156281 -> 156281 filas (0 eliminadas)
   Decisión: trim, espacios múltiples -> uno, MAYÚSCULAS, sin tildes en categorías, 'S. A.' -> 'S.A.', vacíos -> nulo, muestramedica -> booleano.
[4_tipar_fechas] 156281 -> 156281 filas (0 eliminadas)
   Decisión: Las fechas llegan como texto 'MM/DD/YYYY'. Se convierten a date; las no convertibles o futuras imposibles quedan nulas.
[5_tipar_numeros] 156281 -> 156281 filas (0 eliminadas)
   Decisión: expediente, expedientecum y consecutivocu

In [25]:
import pandas as pd, json
pasos = json.load(open("evidencias/curada_pasos.json"))
display(pd.DataFrame(pasos)[["paso", "filas_antes", "filas_despues", "filas_eliminadas", "decision"]])
for p in pasos:
    if "detalle" in p:
        print(p["paso"], "->", json.dumps(p["detalle"], ensure_ascii=False))
print("\nTamaños:", json.dumps(json.load(open("evidencias/curada_tamanos.json")), indent=1, ensure_ascii=False))
rep = json.load(open("data/calidad/reporte_calidad_ultimo.json"))
display(pd.DataFrame(rep["reglas"]))
print("Descartados:", rep["descartados"])
!hdfs dfs -ls /datalake/curada/cum | tee evidencias/hdfs_ls_curada.txt
!hdfs dfs -du -h /datalake | tee evidencias/hdfs_du.txt

paso,filas_antes,filas_despues,filas_eliminadas,decision
0_lectura_raw_hdfs,156281,156281,0,Se leen todos los sobres JSONL del data lake raw.
1_descartar_sin_clave,156281,156281,0,"Se descartan filas sin id_registro, expediente o consecutivocum: sin ellas no se puede identificar ni deduplicar el registro."
2_deduplicar_id_registro,156281,156281,0,Se conserva una fila por id_registro (:id de Socrata): la versión con :updated_at más reciente.
3_normalizar_texto,156281,156281,0,"trim, espacios múltiples -> uno, MAYÚSCULAS, sin tildes en categorías, 'S. A.' -> 'S.A.', vacíos -> nulo, muestramedica -> booleano."
4_tipar_fechas,156281,156281,0,Las fechas llegan como texto 'MM/DD/YYYY'. Se convierten a date; las no convertibles o futuras imposibles quedan nulas.
5_tipar_numeros,156281,156281,0,"expediente, expedientecum y consecutivocum -> long; cantidad y cantidadcum -> double."
6_tratar_nulos,156281,156281,0,Categorías nulas -> 'NO REPORTA'. fechainactivo nula (118.168 filas) SE CONSERVA: significa que el CUM sigue activo.
7_columnas_derivadas,156281,156281,0,"atc_nivel1, grupo_atc, cum_id, anio_expedicion, es_activo, es_importador."


Tamaños: {
 "raw_jsonl_local_bytes": 168783480,
 "raw_jsonl_hdfs_bytes": 168783480,
 "curada_parquet_hdfs_bytes": 14889779,
 "filas_raw_hdfs": 156281,
 "filas_curadas": 156281,
 "reduccion_pct": 91.2,
 "particiones": {
  "A": 28076,
  "B": 15627,
  "C": 18464,
  "D": 4618,
  "G": 7426,
  "H": 4490,
  "J": 12276,
  "L": 7318,
  "M": 12612,
  "N": 27103,
  "P": 1180,
  "R": 10135,
  "S": 2813,
  "V": 4143
 },
 "calidad": "OK"
}


regla,critica,ok,filas_afectadas,detalle
conteo_mayor_que_cero,True,True,0,156281 filas curadas
sin_nulos_en_columnas_clave,True,True,0,"id_registro, expediente, consecutivocum"
id_registro_sin_repetidos,True,True,0,llave de deduplicación
cantidades_en_rango,False,True,0,cantidad >= 0 y cantidadcum > 0
fechaexpedicion_en_rango,False,True,0,entre 1950-01-01 y hoy
estadocum_en_dominio,False,True,0,"valores permitidos: Activo, Inactivo"
coherencia_estado_fechainactivo,False,False,2960,CUM 'Activo' no debería tener fechainactivo (reactivaciones)


Descartados: {'sin_columnas_clave': 0, 'duplicados_id_registro': 0, 'total': 0}
drwxr-xr-x   - root supergroup          0 2026-11-21 14:31 /datalake/curada/cum/atc_nivel1=A
drwxr-xr-x   - root supergroup          0 2026-11-21 14:31 /datalake/curada/cum/atc_nivel1=B
drwxr-xr-x   - root supergroup          0 2026-11-21 14:31 /datalake/curada/cum/atc_nivel1=C
drwxr-xr-x   - root supergroup          0 2026-11-21 14:31 /datalake/curada/cum/atc_nivel1=D
drwxr-xr-x   - root supergroup          0 2026-11-21 14:31 /datalake/curada/cum/atc_nivel1=G
drwxr-xr-x   - root supergroup          0 2026-11-21 14:31 /datalake/curada/cum/atc_nivel1=H
drwxr-xr-x   - root supergroup          0 2026-11-21 14:31 /datalake/curada/cum/atc_nivel1=J
drwxr-xr-x   - root supergroup          0 2026-11-21 14:31 /datalake/curada/cum/atc_nivel1=L
drwxr-xr-x   - root supergroup          0 2026-11-21 14:31 /datalake/curada/cum/atc_nivel1=M
drwxr-xr-x   - root supergroup          0 2026-11-21 14:31 /datalake/curada/cum/atc

## Fase 7. Streaming: capa rápida (Lambda)
Spark Structured Streaming lee el tópico de Kafka y cuenta mensajes por **ventana de 30 s** y tipo de rol,
con **watermark de 2 minutos** y **checkpoint en HDFS**. Cada micro-lote se guarda en SQLite con UPSERT.
Se usa `trigger(availableNow=True)`: procesa lo disponible y se detiene. **Al ejecutarlo otra vez
(reinicio), el checkpoint hace que lea 0 mensajes y los totales no cambian.**

**Unión con el batch (Lambda):** la vista `v_lambda_tiporol` de la zona de servicio compara los registros
de la vista batch (deduplicada) con los mensajes de la capa rápida. Si hubiera duplicados en Kafka, la capa
rápida los contaría y el batch los corregiría: así funciona Lambda, la capa rápida es inmediata pero
aproximada y el batch es exacto.

In [26]:
%%writefile src/06_streaming.py
"""Fase 7 - Capa rápida (Lambda) con Spark Structured Streaming.

Lee el tópico de Kafka como flujo y cuenta mensajes por ventana de tiempo de
30 segundos y por tipo de rol (FABRICANTE / IMPORTADOR), usando:
  - watermark de 2 minutos sobre la marca de tiempo del mensaje en Kafka
    (los eventos que lleguen con más retraso se ignoran y el estado viejo se libera);
  - checkpoint en HDFS: guarda los offsets ya procesados y el estado de la agregación.
    Al reiniciar, el flujo continúa desde donde quedó: NO reprocesa ni duplica.

Cada micro-lote se escribe en SQLite (zona de servicio, tabla stream_conteos_ventana)
con UPSERT por (ventana_inicio, tiporol), así que reescribir un lote es idempotente.

Se usa trigger(availableNow=True): procesa todo lo disponible y se detiene. Correrlo
otra vez equivale a reiniciar el flujo.

Uso:
    python src/06_streaming.py          # procesa lo nuevo y se detiene
    python src/06_streaming.py --reset  # borra checkpoint y tabla (empezar de cero)
"""
import argparse
import sqlite3
import sys

from pyspark.sql import functions as F
from pyspark.sql.types import StringType, StructField, StructType

import config as C
from utils import ahora, guardar_evidencia, hdfs, hdfs_existe

ESQUEMA_MSG = StructType([
    StructField("id_registro", StringType()),
    StructField("fecha_ingesta", StringType()),
    StructField("payload", StructType([
        StructField("tiporol", StringType()),
        StructField("estadocum", StringType()),
        StructField("atc", StringType()),
    ])),
])

DDL = """
CREATE TABLE IF NOT EXISTS stream_conteos_ventana (
    ventana_inicio TEXT NOT NULL,
    ventana_fin    TEXT NOT NULL,
    tiporol        TEXT NOT NULL,
    mensajes       INTEGER NOT NULL,
    actualizado    TEXT NOT NULL,
    PRIMARY KEY (ventana_inicio, tiporol)
)"""


def totales_sqlite():
    con = sqlite3.connect(C.SERVICE_DB)
    con.execute(DDL)
    fila = con.execute("SELECT COUNT(*), COALESCE(SUM(mensajes),0) FROM stream_conteos_ventana").fetchone()
    con.close()
    return {"ventanas": fila[0], "mensajes": fila[1]}


def escribir_lote(batch_df, batch_id):
    filas = batch_df.collect()
    if not filas:
        return
    con = sqlite3.connect(C.SERVICE_DB)
    con.execute(DDL)
    marca = ahora().isoformat()
    con.executemany(
        """INSERT INTO stream_conteos_ventana VALUES (?,?,?,?,?)
           ON CONFLICT(ventana_inicio, tiporol) DO UPDATE SET
             mensajes=excluded.mensajes, actualizado=excluded.actualizado""",
        [(str(r["ventana"]["start"]), str(r["ventana"]["end"]), r["tiporol"],
          int(r["mensajes"]), marca) for r in filas])
    con.commit()
    con.close()
    print(f"   micro-lote {batch_id}: {len(filas)} ventana/s actualizada/s")


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--reset", action="store_true")
    args = ap.parse_args()

    if args.reset:
        if hdfs_existe(C.HDFS_CHECKPOINT):
            hdfs("-rm", "-r", "-skipTrash", C.HDFS_CHECKPOINT)
        con = sqlite3.connect(C.SERVICE_DB)
        con.execute("DROP TABLE IF EXISTS stream_conteos_ventana")
        con.commit()
        con.close()
        print("Checkpoint y tabla de streaming reiniciados.")

    antes = totales_sqlite()
    checkpoint_previo = hdfs_existe(C.HDFS_CHECKPOINT)
    print(f"Checkpoint previo: {'sí (reinicio)' if checkpoint_previo else 'no (primera vez)'}")
    print(f"Antes  -> ventanas: {antes['ventanas']}, mensajes acumulados: {antes['mensajes']}")

    from spark_utils import get_spark
    spark = get_spark("fase7-streaming", kafka=True)

    flujo = (spark.readStream.format("kafka")
             .option("kafka.bootstrap.servers", C.KAFKA_BOOTSTRAP)
             .option("subscribe", C.TOPIC)
             .option("startingOffsets", "earliest")
             .load())

    eventos = (flujo
        .select(F.col("timestamp").alias("ts_kafka"),
                F.from_json(F.col("value").cast("string"), ESQUEMA_MSG).alias("m"))
        .select("ts_kafka",
                F.col("m.id_registro").alias("id_registro"),
                F.coalesce(F.upper(F.trim(F.col("m.payload.tiporol"))),
                           F.lit("NO REPORTA")).alias("tiporol")))

    conteos = (eventos
        .withWatermark("ts_kafka", "2 minutes")
        .groupBy(F.window("ts_kafka", "30 seconds").alias("ventana"), "tiporol")
        .agg(F.count("*").alias("mensajes")))

    consulta = (conteos.writeStream
        .outputMode("update")
        .foreachBatch(escribir_lote)
        .option("checkpointLocation", f"{C.HDFS_URI}{C.HDFS_CHECKPOINT}")
        .trigger(availableNow=True)
        .start())
    consulta.awaitTermination()

    filas_leidas = sum(p.get("numInputRows", 0) for p in consulta.recentProgress)
    lotes = len(consulta.recentProgress)
    spark.stop()

    despues = totales_sqlite()
    evidencia = {
        "fecha": ahora().isoformat(),
        "checkpoint_previo": checkpoint_previo,
        "mensajes_leidos_de_kafka_en_esta_corrida": filas_leidas,
        "micro_lotes": lotes,
        "antes": antes,
        "despues": despues,
    }
    print(f"\nMensajes leídos de Kafka en esta corrida: {filas_leidas} ({lotes} micro-lote/s)")
    print(f"Después -> ventanas: {despues['ventanas']}, mensajes acumulados: {despues['mensajes']}")
    if checkpoint_previo and filas_leidas == 0:
        print("Reinicio verificado: el checkpoint evitó reprocesar mensajes ya leídos.")
    nombre = "streaming_reinicio.json" if checkpoint_previo else "streaming_primera_corrida.json"
    guardar_evidencia(nombre, evidencia)
    return 0


if __name__ == "__main__":
    sys.exit(main())

Writing src/06_streaming.py


In [27]:
!python src/06_streaming.py

Checkpoint previo: no (primera vez)
Antes  -> ventanas: 0, mensajes acumulados: 0
   micro-lote 0: 8 ventana/s actualizada/s

Mensajes leídos de Kafka en esta corrida: 156281 (1 micro-lote/s)
Después -> ventanas: 8, mensajes acumulados: 156281


**Reinicio del flujo** (mismo checkpoint): no debe reprocesar nada.

In [28]:
!python src/06_streaming.py | tee evidencias/streaming_reinicio.txt

Checkpoint previo: sí (reinicio)
Antes  -> ventanas: 8, mensajes acumulados: 156281

Mensajes leídos de Kafka en esta corrida: 0 (0 micro-lote/s)
Después -> ventanas: 8, mensajes acumulados: 156281
Reinicio verificado: el checkpoint evitó reprocesar mensajes ya leídos.


## Fase 8. Zona de servicio y tablero
Los indicadores se calculan en Spark y se guardan en SQLite (`data/servicio/servicio.db`). El tablero
solo los lee. El tablero es un HTML interactivo con Plotly que también se publica en GitHub Pages (`docs/index.html`).

In [29]:
%%writefile src/07_servicio.py
"""Fase 8a - Zona de servicio: indicadores listos para el tablero en SQLite.

Lee la zona curada (Parquet en HDFS), calcula los indicadores que responden la
pregunta y los guarda en data/servicio/servicio.db. El tablero solo lee estas tablas
y no hace cálculos. Cada tabla se reemplaza completa en cada corrida (idempotente).

Unidad de análisis: el CUM (cum_id = expedientecum-consecutivocum), es decir, cada
presentación comercial. Una fila curada es CUM x principio activo x rol, por eso se
cuentan CUM distintos.
"""
import sqlite3
import sys

from pyspark.sql import functions as F

import config as C
from spark_utils import get_spark
from utils import ahora, guardar_evidencia

DDL_STREAM = """
CREATE TABLE IF NOT EXISTS stream_conteos_ventana (
    ventana_inicio TEXT NOT NULL, ventana_fin TEXT NOT NULL, tiporol TEXT NOT NULL,
    mensajes INTEGER NOT NULL, actualizado TEXT NOT NULL,
    PRIMARY KEY (ventana_inicio, tiporol))"""

VISTA_LAMBDA = """
CREATE VIEW v_lambda_tiporol AS
SELECT b.tiporol,
       b.registros AS registros_batch,
       COALESCE(s.mensajes, 0) AS mensajes_stream,
       COALESCE(s.mensajes, 0) - b.registros AS diferencia
FROM ind_registros_tiporol b
LEFT JOIN (SELECT tiporol, SUM(mensajes) AS mensajes
           FROM stream_conteos_ventana GROUP BY tiporol) s
       ON s.tiporol = b.tiporol"""


def main():
    spark = get_spark("fase8-servicio")
    cur = spark.read.parquet(f"{C.HDFS_URI}{C.HDFS_CURADA}").cache()
    activos = cur.filter("es_activo")
    tablas = {}

    # 1. CUM activos / inactivos por grupo ATC
    tablas["ind_atc_estado"] = (cur.groupBy("atc_nivel1", "grupo_atc").agg(
        F.countDistinct(F.when(F.col("es_activo"), F.col("cum_id"))).alias("cum_activos"),
        F.countDistinct(F.when(~F.col("es_activo"), F.col("cum_id"))).alias("cum_inactivos"),
        F.countDistinct("cum_id").alias("cum_total"))
        .orderBy(F.desc("cum_activos")))

    # 2. Dependencia de importación por grupo ATC (CUM activos con rol importador)
    tablas["ind_origen_atc"] = (activos.groupBy("atc_nivel1", "grupo_atc").agg(
        F.countDistinct("cum_id").alias("cum_activos"),
        F.countDistinct(F.when(F.col("es_importador"), F.col("cum_id"))).alias("cum_con_importador"),
        F.countDistinct(F.when(~F.col("es_importador"), F.col("cum_id"))).alias("cum_con_fabricante"))
        .withColumn("pct_importador", F.round(100 * F.col("cum_con_importador") / F.col("cum_activos"), 1))
        .orderBy(F.desc("pct_importador")))

    # 3. Concentración: top 15 titulares por CUM activos
    total_activos = activos.select("cum_id").distinct().count()
    top = (activos.groupBy("titular").agg(F.countDistinct("cum_id").alias("cum_activos"))
           .orderBy(F.desc("cum_activos"), "titular").limit(15).toPandas())
    top["pct_del_total"] = (100 * top["cum_activos"] / total_activos).round(2)
    top["pct_acumulado"] = top["pct_del_total"].cumsum().round(2)
    top.insert(0, "posicion", range(1, len(top) + 1))

    # 4. Evolución: registros sanitarios vigentes por año de expedición e inactivaciones por año
    exp = (cur.filter(F.col("anio_expedicion") >= 1990).groupBy(F.col("anio_expedicion").alias("anio"))
           .agg(F.countDistinct("expediente").alias("registros_expedidos")))
    ina = (cur.filter(F.col("fechainactivo").isNotNull())
           .groupBy(F.year("fechainactivo").alias("anio"))
           .agg(F.countDistinct("cum_id").alias("cum_inactivados")))
    tablas["ind_evolucion_anual"] = (exp.join(ina, "anio", "full").fillna(0)
                                     .filter(F.col("anio") >= 1990).orderBy("anio"))

    # 5. Vía de administración (CUM activos)
    tablas["ind_via_administracion"] = (activos.groupBy("viaadministracion")
        .agg(F.countDistinct("cum_id").alias("cum_activos")).orderBy(F.desc("cum_activos")))

    # 6. Registros (filas curadas) por tipo de rol: vista batch para unir con el streaming
    tablas["ind_registros_tiporol"] = cur.groupBy("tiporol").agg(F.count("*").alias("registros"))

    # 7. Resumen (KPIs) y fecha de corte
    cum_imp = activos.filter("es_importador").select("cum_id").distinct().count()
    fila = cur.agg(
        F.countDistinct("cum_id").alias("cum_total"),
        F.countDistinct(F.when(F.col("es_activo"), F.col("cum_id"))).alias("cum_activos"),
        F.countDistinct("expediente").alias("registros_sanitarios"),
        F.countDistinct(F.when(F.col("es_activo"), F.col("titular"))).alias("titulares_activos"),
        F.countDistinct("principioactivo").alias("principios_activos"),
        F.max("actualizado_fuente").alias("fecha_corte_fuente"),
        F.count("*").alias("filas_curadas")).first().asDict()
    pct = (top["cum_activos"] / total_activos)
    resumen = {
        **fila,
        "cum_inactivos": fila["cum_total"] - fila["cum_activos"],
        "pct_cum_activos_con_importador": round(100 * cum_imp / total_activos, 1),
        "pct_top10_titulares": round(100 * top["cum_activos"].head(10).sum() / total_activos, 1),
        "fuente": f"{C.NOMBRE_DATASET} ({C.FUENTE}, dataset {C.DATASET})",
        "url_fuente": C.API_URL,
        "fecha_proceso": ahora().isoformat(),
    }
    print("Resumen:", resumen)

    # ---------- Escritura en SQLite ----------
    import pandas as pd
    con = sqlite3.connect(C.SERVICE_DB)
    con.execute("DROP VIEW IF EXISTS v_lambda_tiporol")
    conteos = {}
    for nombre, sdf in tablas.items():
        pdf = sdf.toPandas()
        pdf.to_sql(nombre, con, if_exists="replace", index=False)
        conteos[nombre] = len(pdf)
    top.to_sql("ind_top_titulares", con, if_exists="replace", index=False)
    conteos["ind_top_titulares"] = len(top)
    pd.DataFrame([resumen]).to_sql("ind_resumen", con, if_exists="replace", index=False)
    conteos["ind_resumen"] = 1
    con.execute(DDL_STREAM)
    con.execute(VISTA_LAMBDA)
    con.commit()
    print("\nVista Lambda (batch vs. capa rápida):")
    for r in con.execute("SELECT * FROM v_lambda_tiporol"):
        print("  ", r)
    con.close()
    spark.stop()

    for t, n in conteos.items():
        print(f"  {t:<26} {n:>5} filas")
    guardar_evidencia("servicio_tablas.json", {"tablas": conteos, "resumen": resumen})
    return 0


if __name__ == "__main__":
    sys.exit(main())

Writing src/07_servicio.py


In [30]:
!python src/07_servicio.py

Resumen: {'cum_total': 65441, 'cum_activos': 50255, 'registros_sanitarios': 9500, 'titulares_activos': 1159, 'principios_activos': 5931, 'fecha_corte_fuente': '2026-11-16T12:13:21.891Z', 'filas_curadas': 156281, 'cum_inactivos': 15186, 'pct_cum_activos_con_importador': 38.6, 'pct_top10_titulares': 31.7, 'fuente': 'Código Único de Medicamentos (CUM) vigentes - INVIMA (datos.gov.co, dataset i7cb-raxc)', 'url_fuente': 'https://www.datos.gov.co/resource/i7cb-raxc.json', 'fecha_proceso': '2026-11-21T14:32:00-05:00'}

Vista Lambda (batch vs. capa rápida):
   ('FABRICANTE', 114606, 114606, 0)
   ('IMPORTADOR', 41675, 41675, 0)
  ind_atc_estado                14 filas
  ind_origen_atc                14 filas
  ind_top_titulares             15 filas
  ind_evolucion_anual           37 filas
  ind_via_administracion        47 filas
  ind_registros_tiporol          2 filas
  ind_resumen                    1 filas


In [31]:
import sqlite3, pandas as pd
con = sqlite3.connect("data/servicio/servicio.db")
for t in ["ind_resumen", "ind_atc_estado", "ind_origen_atc", "ind_top_titulares", "v_lambda_tiporol"]:
    print("==", t); display(pd.read_sql(f"SELECT * FROM {t}", con).head(15))

== ind_resumen


cum_total,cum_activos,registros_sanitarios,titulares_activos,principios_activos,fecha_corte_fuente,filas_curadas,cum_inactivos,pct_cum_activos_con_importador,pct_top10_titulares,fuente,url_fuente,fecha_proceso
65441,50255,9500,1159,5931,2026-11-16T12:13:21.891Z,156281,15186,38.6,31.7,"Código Único de Medicamentos (CUM) vigentes - INVIMA (datos.gov.co, dataset i7cb-raxc)",https://www.datos.gov.co/resource/i7cb-raxc.json,2026-11-21T14:32:00-05:00


== ind_atc_estado


atc_nivel1,grupo_atc,cum_activos,cum_inactivos,cum_total
A,Tracto alimentario y metabolismo,8740,2213,10953
N,Sistema nervioso,8572,3034,11606
C,Sistema cardiovascular,6702,2460,9162
M,Sistema musculoesquelético,4206,1313,5519
J,Antiinfecciosos sistémicos,4151,1445,5596
R,Sistema respiratorio,3844,1210,5054
B,Sangre y órganos hematopoyéticos,2646,635,3281
L,Antineoplásicos e inmunomoduladores,2645,439,3084
G,Sistema genitourinario y hormonas sexuales,2470,732,3202
D,Dermatológicos,2054,589,2643


== ind_origen_atc


atc_nivel1,grupo_atc,cum_activos,cum_con_importador,cum_con_fabricante,pct_importador
L,Antineoplásicos e inmunomoduladores,2645,2620,2620,99.1
B,Sangre y órganos hematopoyéticos,2646,1418,2644,53.6
G,Sistema genitourinario y hormonas sexuales,2470,1159,2467,46.9
N,Sistema nervioso,8572,3857,8570,45.0
C,Sistema cardiovascular,6702,2491,6689,37.2
A,Tracto alimentario y metabolismo,8740,3005,8734,34.4
J,Antiinfecciosos sistémicos,4151,1380,4142,33.2
V,Varios,1248,389,1248,31.2
M,Sistema musculoesquelético,4206,1254,4206,29.8
S,Órganos de los sentidos,971,282,971,29.0


== ind_top_titulares


posicion,titular,cum_activos,pct_del_total,pct_acumulado
1,NOVAMED S.A.S.,3829,7.62,7.62
2,TECNOQUIMICAS S.A.,2677,5.33,12.95
3,PROCAPS S.A.,1768,3.52,16.47
4,LABORATORIOS SIEGFRIED S.A.S.,1380,2.75,19.22
5,LABORATORIOS ECAR S.A.,1318,2.62,21.84
6,LABORATORIOS LEGRAND S.A.,1208,2.4,24.24
7,MEGALABS COLOMBIA S.A.S,1020,2.03,26.27
8,LABORATORIO FRANCO COLOMBIANO - LAFRANCOL S.A.S.,937,1.86,28.13
9,PFIZER S.A.S.,904,1.8,29.93
10,EUROFARMA COLOMBIA S.A.S.,893,1.78,31.71


== v_lambda_tiporol


tiporol,registros_batch,mensajes_stream,diferencia
FABRICANTE,114606,114606,0
IMPORTADOR,41675,41675,0


In [32]:
%%writefile src/08_tablero.py
"""Fase 8b - Tablero HTML interactivo (Plotly) que lee SOLO la zona de servicio.

Genera dashboard/tablero.html y docs/index.html (para publicarlo gratis con
GitHub Pages). No hace cálculos: solo lee las tablas ind_* de SQLite.
"""
import os
import sqlite3
import sys
from datetime import datetime

import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio

import config as C

COLOR = {"primario": "#0B5394", "secundario": "#E69138", "gris": "#B7B7B7",
         "verde": "#38761D", "fondo": "#F5F7FA"}
pio.templates.default = "plotly_white"
FUENTE_FIG = dict(family="Inter, Segoe UI, Arial", size=13, color="#1F2937")


def leer(con, tabla):
    try:
        return pd.read_sql(f"SELECT * FROM {tabla}", con)
    except Exception:
        return pd.DataFrame()


def estilo(fig, alto=420):
    fig.update_layout(height=alto, margin=dict(l=10, r=10, t=10, b=10), font=FUENTE_FIG,
                      legend=dict(orientation="h", y=-0.15), paper_bgcolor="white",
                      plot_bgcolor="white")
    return fig


def a_html(fig):
    return pio.to_html(fig, include_plotlyjs=False, full_html=False,
                       config={"displayModeBar": False, "responsive": True})


def fmt(n):
    return f"{int(n):,}".replace(",", ".")


def main():
    con = sqlite3.connect(C.SERVICE_DB)
    r = leer(con, "ind_resumen").iloc[0]
    atc = leer(con, "ind_atc_estado")
    origen = leer(con, "ind_origen_atc")
    top = leer(con, "ind_top_titulares")
    evo = leer(con, "ind_evolucion_anual")
    via = leer(con, "ind_via_administracion")
    stream = leer(con, "stream_conteos_ventana")
    lam = leer(con, "v_lambda_tiporol")
    con.close()

    # --- 1. Oferta por grupo ATC (activos vs inactivos) ---
    a = atc[atc.atc_nivel1 != "SIN_ATC"].sort_values("cum_activos")
    f1 = go.Figure([
        go.Bar(y=a.grupo_atc, x=a.cum_activos, name="Activos", orientation="h",
               marker_color=COLOR["primario"]),
        go.Bar(y=a.grupo_atc, x=a.cum_inactivos, name="Inactivos", orientation="h",
               marker_color=COLOR["gris"])])
    f1.update_layout(barmode="stack", xaxis_title="Número de CUM (presentaciones)")
    estilo(f1, 480)

    # --- 2. Dependencia de importación por grupo ATC ---
    o = origen[origen.atc_nivel1 != "SIN_ATC"].sort_values("pct_importador")
    f2 = go.Figure(go.Bar(
        y=o.grupo_atc, x=o.pct_importador, orientation="h",
        marker=dict(color=o.pct_importador, colorscale="Oranges", cmin=0, cmax=100),
        text=[f"{v:.0f} %" for v in o.pct_importador], textposition="outside",
        customdata=o[["cum_con_importador", "cum_activos"]],
        hovertemplate="%{y}<br>%{x:.1f} % importado<br>%{customdata[0]} de %{customdata[1]} CUM<extra></extra>"))
    f2.update_layout(xaxis=dict(title="% de CUM activos con rol importador", range=[0, 105]))
    estilo(f2, 480)

    # --- 3. Concentración: top titulares ---
    t = top.sort_values("cum_activos")
    f3 = go.Figure(go.Bar(
        y=t.titular, x=t.cum_activos, orientation="h", marker_color=COLOR["primario"],
        text=[f"{p:.1f} %" for p in t.pct_del_total], textposition="outside",
        hovertemplate="%{y}<br>%{x} CUM activos<extra></extra>"))
    f3.update_layout(xaxis_title="CUM activos")
    estilo(f3, 520)

    # --- 4. Evolución anual ---
    f4 = go.Figure([
        go.Bar(x=evo.anio, y=evo.registros_expedidos, name="Registros sanitarios vigentes expedidos",
               marker_color=COLOR["primario"]),
        go.Scatter(x=evo.anio, y=evo.cum_inactivados, name="CUM inactivados", mode="lines+markers",
                   line=dict(color=COLOR["secundario"], width=3), yaxis="y2")])
    f4.update_layout(yaxis=dict(title="Registros expedidos"),
                     yaxis2=dict(title="CUM inactivados", overlaying="y", side="right",
                                 showgrid=False), xaxis_title="Año")
    estilo(f4, 420)

    # --- 5. Vía de administración ---
    v = via.copy()
    if len(v) > 6:
        otras = pd.DataFrame([{"viaadministracion": "OTRAS",
                               "cum_activos": v.cum_activos.iloc[6:].sum()}])
        v = pd.concat([v.head(6), otras])
    f5 = go.Figure(go.Pie(labels=v.viaadministracion, values=v.cum_activos, hole=0.55,
                          sort=False, textinfo="percent",
                          marker=dict(colors=["#0B5394", "#3D85C6", "#6FA8DC", "#9FC5E8",
                                              "#E69138", "#F6B26B", "#CCCCCC"])))
    estilo(f5, 420)

    # --- 6. Capa rápida (streaming) + vista Lambda ---
    if not stream.empty:
        s = stream.sort_values("ventana_inicio")
        f6 = go.Figure([go.Bar(x=g.ventana_inicio.str.slice(11, 19), y=g.mensajes, name=rol)
                        for rol, g in s.groupby("tiporol")])
        f6.update_layout(barmode="stack", xaxis_title="Ventana de 30 s (hora del mensaje en Kafka)",
                         yaxis_title="Mensajes")
        estilo(f6, 360)
        g6 = a_html(f6)
    else:
        g6 = "<p>Aún no hay datos de streaming.</p>"
    tabla_lambda = lam.to_html(index=False, classes="tabla", border=0) if not lam.empty else ""

    # --- Título que responde la pregunta ---
    g = atc[atc.atc_nivel1 != "SIN_ATC"].sort_values("cum_activos", ascending=False)
    g1, g2 = g.grupo_atc.iloc[0].lower(), g.grupo_atc.iloc[1].lower()
    titulo = (f"{r.pct_cum_activos_con_importador:.0f} % de los medicamentos activos en Colombia "
              f"depende de importadores; {g1} y {g2} concentran la oferta")
    corte = str(r.fecha_corte_fuente)[:10]
    proceso = str(r.fecha_proceso)[:16].replace("T", " ")

    kpis = [
        ("CUM activos", fmt(r.cum_activos), "presentaciones comerciales vigentes"),
        ("CUM inactivos", fmt(r.cum_inactivos), "presentaciones dadas de baja"),
        ("% con importador", f"{r.pct_cum_activos_con_importador:.1f} %", "de los CUM activos"),
        ("Top 10 titulares", f"{r.pct_top10_titulares:.1f} %", "de los CUM activos"),
        ("Titulares", fmt(r.titulares_activos), "con al menos un CUM activo"),
        ("Principios activos", fmt(r.principios_activos), "distintos"),
    ]
    kpi_html = "".join(f'<div class="kpi"><div class="k-t">{a}</div><div class="k-v">{b}</div>'
                       f'<div class="k-s">{c}</div></div>' for a, b, c in kpis)

    def tarjeta(titulo_t, sub, graf, ancho=""):
        return (f'<section class="card {ancho}"><h3>{titulo_t}</h3><p class="sub">{sub}</p>'
                f'{graf}</section>')

    html = f"""<!DOCTYPE html>
<html lang="es"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Medicamentos vigentes en Colombia - Tablero CUM INVIMA</title>
<script src="https://cdn.plot.ly/plotly-2.35.2.min.js"></script>
<link href="https://fonts.googleapis.com/css2?family=Inter:wght@400;600;800&display=swap" rel="stylesheet">
<style>
 body{{margin:0;font-family:Inter,Segoe UI,Arial,sans-serif;background:{COLOR['fondo']};color:#1F2937}}
 header{{background:linear-gradient(120deg,#073763,#0B5394 60%,#3D85C6);color:white;padding:34px 48px 28px}}
 header .eti{{text-transform:uppercase;letter-spacing:.12em;font-size:12px;opacity:.8}}
 header h1{{margin:8px 0 10px;font-size:30px;line-height:1.25;font-weight:800;max-width:1100px}}
 header p{{margin:0;opacity:.9;max-width:1000px}}
 main{{padding:24px 48px 10px;max-width:1500px;margin:auto}}
 .kpis{{display:grid;grid-template-columns:repeat(auto-fit,minmax(190px,1fr));gap:14px;margin-top:-50px}}
 .kpi{{background:white;border-radius:14px;padding:16px 18px;box-shadow:0 4px 18px rgba(0,0,0,.08)}}
 .k-t{{font-size:12px;text-transform:uppercase;color:#6B7280;letter-spacing:.06em}}
 .k-v{{font-size:28px;font-weight:800;color:#0B5394;margin:4px 0}}
 .k-s{{font-size:12px;color:#6B7280}}
 .grid{{display:grid;grid-template-columns:1fr 1fr;gap:18px;margin-top:20px}}
 .card{{background:white;border-radius:14px;padding:18px 20px;box-shadow:0 2px 10px rgba(0,0,0,.06)}}
 .card.full{{grid-column:1/-1}}
 .card h3{{margin:0;font-size:17px}} .sub{{margin:4px 0 8px;color:#6B7280;font-size:13px}}
 .tabla{{border-collapse:collapse;font-size:13px;margin-top:10px}}
 .tabla th,.tabla td{{padding:6px 12px;border-bottom:1px solid #E5E7EB;text-align:right}}
 .tabla th{{background:#F3F4F6}}
 footer{{padding:18px 48px 30px;color:#4B5563;font-size:13px;max-width:1500px;margin:auto}}
 @media(max-width:900px){{.grid{{grid-template-columns:1fr}} header,main,footer{{padding-left:18px;padding-right:18px}}}}
</style></head><body>
<header>
 <div class="eti">Gestión de Datos Masivos · Pipeline CUM INVIMA</div>
 <h1>{titulo}</h1>
 <p>¿Cómo está compuesta la oferta de medicamentos con registro vigente en Colombia por grupo
 terapéutico, vía y origen, y qué tanto depende de importaciones y de pocos titulares?</p>
 <br><br>
</header>
<main>
 <div class="kpis">{kpi_html}</div>
 <div class="grid">
  {tarjeta("Oferta por grupo terapéutico (ATC)", "CUM activos e inactivos por grupo ATC nivel 1", a_html(f1))}
  {tarjeta("Dependencia de importación", "% de CUM activos con rol de importador, por grupo ATC", a_html(f2))}
  {tarjeta("Concentración en pocos titulares", f"Top 15 titulares por CUM activos · los 10 primeros suman {r.pct_top10_titulares:.1f} %", a_html(f3))}
  {tarjeta("Vía de administración", "Distribución de los CUM activos", a_html(f5))}
  {tarjeta("Evolución en el tiempo", "Registros sanitarios vigentes por año de expedición y CUM inactivados por año", a_html(f4), "full")}
  {tarjeta("Capa rápida (streaming) · arquitectura Lambda", "Mensajes recibidos por Kafka en ventanas de 30 s (Spark Structured Streaming). La tabla compara la vista batch (deduplicada) con la vista rápida.", g6 + tabla_lambda, "full")}
 </div>
</main>
<footer>
 <b>Fuente:</b> {r.fuente} · API: <a href="{r.url_fuente}">{r.url_fuente}</a><br>
 <b>Fecha de corte de los datos:</b> {corte} (última actualización de registros en la fuente) ·
 <b>Procesado:</b> {proceso} · Unidad de análisis: CUM (expediente + consecutivo).<br>
 <b>Autores:</b> Martín Sánchez Urrego · Samuel Montoya Garzón — Institución Universitaria Pascual Bravo (Gestión de Datos Masivos G002, 2026-2).
</footer>
</body></html>"""

    salidas = [os.path.join(C.BASE_DIR, "dashboard", "tablero.html"),
               os.path.join(C.BASE_DIR, "docs", "index.html")]
    for ruta in salidas:
        os.makedirs(os.path.dirname(ruta), exist_ok=True)
        with open(ruta, "w", encoding="utf-8") as f:
            f.write(html)
        print("Tablero generado:", ruta)
    print("Título:", titulo)
    return 0


if __name__ == "__main__":
    sys.exit(main())

Writing src/08_tablero.py


In [33]:
!python src/08_tablero.py

Tablero generado: /content/pp-cum-invima/dashboard/tablero.html
Tablero generado: /content/pp-cum-invima/docs/index.html
Título: 39 % de los medicamentos activos en Colombia depende de importadores; sistema nervioso y tracto alimentario y metabolismo concentran la oferta


In [34]:
from IPython.display import IFrame, HTML, display
display(HTML(open("dashboard/tablero.html", encoding="utf-8").read()))

tiporol,registros_batch,mensajes_stream,diferencia
FABRICANTE,114606,114606,0
IMPORTADOR,41675,41675,0


## Demostración de idempotencia (fase 6): el pipeline completo se corre dos veces
`run_pipeline.py` ejecuta las fases 2 a 8 en orden y guarda los conteos de cada corrida. En ambas
corridas la API devuelve 0 registros nuevos (marca de agua), Kafka no recibe mensajes y los conteos
de raw, curada, servicio y streaming quedan **idénticos**.

In [35]:
%%writefile src/run_pipeline.py
"""Orquestador: corre el pipeline completo (fases 2 a 8) y registra conteos.

Sirve para demostrar la idempotencia (fase 6): si se ejecuta dos veces seguidas, la
segunda corrida no trae registros nuevos (marca de agua), no publica nada en Kafka y
los conteos de la zona curada y de servicio quedan iguales.

Uso:
    python src/run_pipeline.py --etiqueta corrida_1
    python src/run_pipeline.py --etiqueta corrida_2
    PP_SIMULAR_FALLO=1 python src/run_pipeline.py --etiqueta fallo   # se detiene en calidad
"""
import argparse
import json
import os
import sqlite3
import subprocess
import sys
import time

import config as C
from utils import (ahora, archivos_raw_locales, hdfs_contar_lineas, leer_json,
                   ultimo_manifiesto)

AQUI = os.path.dirname(os.path.abspath(__file__))
PASOS = [
    ("Fase 2   - Ingesta API -> raw", "01_ingesta.py"),
    ("Fase 3   - Productor Kafka", "02_productor.py"),
    ("Fase 3/4 - Consumidor Kafka -> HDFS", "03_consumidor_hdfs.py"),
    ("Fase 4   - MapReduce vs Spark", "04_mapreduce.py"),
    ("Fase 5/6 - Zona curada + calidad", "05_curada_spark.py"),
    ("Fase 7   - Streaming (capa rápida)", "06_streaming.py"),
    ("Fase 8   - Zona de servicio", "07_servicio.py"),
    ("Fase 8   - Tablero", "08_tablero.py"),
]
REGISTRO = os.path.join(C.EVIDENCIAS_DIR, "idempotencia_corridas.jsonl")


def offsets_kafka():
    from confluent_kafka import Consumer, TopicPartition
    c = Consumer({"bootstrap.servers": C.KAFKA_BOOTSTRAP, "group.id": "conteo-offsets"})
    md = c.list_topics(C.TOPIC, timeout=10).topics[C.TOPIC]
    total = sum(c.get_watermark_offsets(TopicPartition(C.TOPIC, p), timeout=10)[1]
                for p in md.partitions)
    c.close()
    return total


def conteos(etiqueta, segundos):
    man = ultimo_manifiesto() or {}
    raw_local = 0
    for a in archivos_raw_locales():
        with open(a, encoding="utf-8") as f:
            raw_local += sum(1 for _ in f)
    calidad = leer_json(os.path.join(C.CALIDAD_DIR, "reporte_calidad_ultimo.json"), {})
    con = sqlite3.connect(C.SERVICE_DB)
    cum_total = con.execute("SELECT cum_total FROM ind_resumen").fetchone()[0]
    stream = con.execute("SELECT COALESCE(SUM(mensajes),0) FROM stream_conteos_ventana").fetchone()[0]
    con.close()
    return {
        "etiqueta": etiqueta,
        "fecha": ahora().isoformat(timespec="seconds"),
        "tipo_carga": man.get("tipo_carga"),
        "registros_nuevos_api": man.get("total_registros"),
        "lineas_raw_local": raw_local,
        "mensajes_en_kafka": offsets_kafka(),
        "lineas_raw_hdfs": hdfs_contar_lineas(f"{C.HDFS_RAW}/fecha_ingesta=*/*.jsonl"),
        "filas_curadas": calidad.get("filas_curadas"),
        "cum_en_servicio": cum_total,
        "mensajes_streaming": stream,
        "watermark": leer_json(C.WATERMARK_FILE, {}).get("watermark"),
        "duracion_s": round(segundos, 1),
    }


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--etiqueta", default=ahora().strftime("corrida_%H%M%S"))
    ap.add_argument("--sin-mapreduce", action="store_true", help="omitir la fase 4 (más rápido)")
    args = ap.parse_args()

    t0 = time.time()
    for nombre, script in PASOS:
        if args.sin_mapreduce and script == "04_mapreduce.py":
            continue
        print(f"\n{'=' * 70}\n>> {nombre}\n{'=' * 70}", flush=True)
        r = subprocess.run([sys.executable, os.path.join(AQUI, script)])
        if r.returncode != 0:
            print(f"\nPIPELINE DETENIDO en '{nombre}' (código {r.returncode}). "
                  "Los pasos siguientes no se ejecutaron.")
            return r.returncode

    fila = conteos(args.etiqueta, time.time() - t0)
    with open(REGISTRO, "a", encoding="utf-8") as f:
        f.write(json.dumps(fila, ensure_ascii=False) + "\n")

    import pandas as pd
    with open(REGISTRO, encoding="utf-8") as f:
        historial = pd.DataFrame([json.loads(l) for l in f if l.strip()])
    print(f"\n{'=' * 70}\nConteos por corrida (evidencias/idempotencia_corridas.jsonl)\n{'=' * 70}")
    with pd.option_context("display.max_columns", None, "display.width", 200):
        print(historial.drop(columns=["fecha"]).tail(6).to_string(index=False))
    return 0


if __name__ == "__main__":
    sys.exit(main())

Writing src/run_pipeline.py


In [36]:
!python src/run_pipeline.py --etiqueta corrida_1

Conteos por corrida (evidencias/idempotencia_corridas.jsonl)
 etiqueta  tipo_carga  registros_nuevos_api  lineas_raw_local  mensajes_en_kafka  lineas_raw_hdfs  filas_curadas  cum_en_servicio  mensajes_streaming                watermark  duracion_s
corrida_1    completa                156281            156281             156281           156281         156281            65441              156281 2026-11-16T12:13:21.891Z        94.2


In [37]:
!python src/run_pipeline.py --etiqueta corrida_2 | tee evidencias/idempotencia_salida.txt

Conteos por corrida (evidencias/idempotencia_corridas.jsonl)
 etiqueta  tipo_carga  registros_nuevos_api  lineas_raw_local  mensajes_en_kafka  lineas_raw_hdfs  filas_curadas  cum_en_servicio  mensajes_streaming                watermark  duracion_s
corrida_1    completa                156281            156281             156281           156281         156281            65441              156281 2026-11-16T12:13:21.891Z        94.2
corrida_2 incremental                     0            156281             156281           156281         156281            65441              156281 2026-11-16T12:13:21.891Z        24.8


**Demostración: una validación crítica falla y el pipeline se detiene.** Se inyectan duplicados con
`PP_SIMULAR_FALLO=1`: la regla `id_registro_sin_repetidos` falla, el pipeline se detiene en la fase 5 y
no se escribe nada en la zona curada ni en la de servicio.

In [38]:
!PP_SIMULAR_FALLO=1 python src/run_pipeline.py --etiqueta fallo_simulado --sin-mapreduce | tee evidencias/fallo_calidad_simulado.txt

*** SIMULACIÓN DE FALLO: se inyectan 10 filas duplicadas ***

=== Validaciones de calidad ===
  [OK ] conteo_mayor_que_cero: 156291 filas
  [OK ] sin_nulos_en_columnas_clave: 0 fila/s afectada/s. id_registro, expediente, consecutivocum
  [FALLA CRÍTICA] id_registro_sin_repetidos: 10 fila/s afectada/s. llave de deduplicación
  Reporte de calidad: data/calidad/reporte_calidad_20261121T143600.json -> FALLA

 PIPELINE DETENIDO: Validaciones críticas fallidas: ['id_registro_sin_repetidos']
   No se escribió la zona curada y la marca de agua NO se movió.

PIPELINE DETENIDO en 'Fase 5/6 - Zona curada + calidad' (código 2). Los pasos siguientes no se ejecutaron.


In [39]:
# El reporte de calidad registra la falla. Volvemos a correr el paso de curada para dejar el estado sano.
!python src/05_curada_spark.py > /dev/null && echo "Zona curada restaurada"

Zona curada restaurada


## Empaquetar evidencias y datos para el repositorio

In [40]:
!mkdir -p data/muestra_raw
!head -n 1000 $(ls data/raw/datos.gov.co/i7cb-raxc/fecha_ingesta=*/*.jsonl | head -1) > data/muestra_raw/muestra_1000.jsonl
import sqlite3, pandas as pd, os
con = sqlite3.connect("data/servicio/servicio.db")
os.makedirs("data/servicio/csv", exist_ok=True)
for (t,) in con.execute("SELECT name FROM sqlite_master WHERE type IN ('table','view')"):
    pd.read_sql(f"SELECT * FROM {t}", con).to_csv(f"data/servicio/csv/{t}.csv", index=False)
!zip -qr /content/entrega_pp.zip evidencias data/manifiestos data/calidad data/servicio data/muestra_raw dashboard docs src setup_colab.sh requirements.txt
from google.colab import files
files.download("/content/entrega_pp.zip")

Descargando /content/entrega_pp.zip ...
